<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇳🇴 Norway Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: developer.entur.org · Provider: Entur (Norwegian National Access Point) · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

Norway is one of the most straightforward countries for data access. Both GTFS and NeTEx are freely available from a single official source which is the Entur developer portal with no registration required. The page clearly lists both formats for the entire country as a single national aggregated file, making it very easy to find and download.

**Norway GTFS Dataset**
- Provider: Entur (Norwegian National Access Point)
- Publisher: Entur AS — owned by the Norwegian Ministry of Transport
- Format: GTFS
- Dataset page: https://developer.entur.org/stops-and-timetable-data
- Download URL: https://storage.googleapis.com/marduk-production/outbound/gtfs/rb_norway-aggregated-gtfs.zip
- Accessed: May 2026
- License: NLOD (Norwegian Licence for Open Government Data)

**Norway NeTEx Dataset**
- Provider: Entur (Norwegian National Access Point)
- Publisher: Entur AS — owned by the Norwegian Ministry of Transport
- Format: NeTEx (Nordic NeTEx Profile)
- Dataset page: https://developer.entur.org/stops-and-timetable-data
- Download URL: https://storage.googleapis.com/marduk-production/outbound/netex/rb_norway-aggregated-netex.zip
- Accessed: May 2026
- License: NLOD (Norwegian Licence for Open Government Data)

*No registration needed for downloading either dataset*

## Table of Contents

**Data Source**

**Part 1: GTFS Exploration**
- Inspect the loaded GTFS tables
- GTFS stop hierarchy
- Build GTFS station-level table
- Prepare GTFS station-level dataset
- GTFS routes
- GTFS transport modes
- Prepare GTFS line-level dataset
- Inspect routes without public line labels
- GTFS calendar / validity level
- Check GTFS service_id coverage
- Rebuild GTFS active service days

**Part 2: NeTEx Exploration**
- Inspect NeTEx XML structure
- Inspect a sample of NeTEx StopPlace elements
- Extract all NeTEx StopPlaces
- Inspect NeTEx line structure
- Extract all NeTEx Lines
- Inspect NeTEx lines without public codes
- Prepare NeTEx line-level dataset
- Inspect NeTEx calendar structure
- Inspect NeTEx calendar element samples
- Extract NeTEx DayTypeAssignments
- Extract NeTEx OperatingPeriods
- Build NeTEx active dates
- Prepare NeTEx calendar summary

**Part 3: GTFS–NeTEx Comparison**
- 3.1 Station-level comparison
- 3.2 Line-level comparison
  - Line-level comparison: GTFS vs NeTEx
  - Check transport modes for matched line labels
- 3.3 Calendar comparison
  - Calendar-level comparison: GTFS vs NeTEx (ID overlap)
  - Calendar-level comparison for matched IDs
  - Calendar-level pattern comparison within the shared date window
  - Build calendar activity patterns in the shared window
  - Compare unique calendar activity patterns
  - Note on comparison method
  - Calendar-level comparison: approach and results
  - Conclusion: calendar-level MVD for Norway

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import defaultdict
import xml.etree.ElementTree as ET
from collections import Counter

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "rb_norway-aggregated-gtfs.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/rb_norway-aggregated-gtfs.zip


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
3,shapes.txt,2716.64
7,stop_times.txt,870.58
2,trips.txt,64.69
4,calendar_dates.txt,32.52
5,stops.txt,9.77
6,transfers.txt,8.58
8,routes.txt,0.30
1,calendar.txt,0.26
9,agency.txt,0.01
0,feed_info.txt,0.00


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:", stops.shape)
print("routes:", routes.shape)
print("trips:", trips.shape)
print("calendar:", calendar.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (142702, 11)
routes: (4420, 9)
trips: (407819, 7)
calendar: (3798, 10)
calendar_dates: (290234, 3)


## Comment

The dataset is large, especially for stops, trips, and calendar_dates, but the necessary tables for station, line, and calendar analysis are available.

## Inspect the loaded GTFS tables

In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_desc', 'location_type', 'parent_station', 'wheelchair_boarding', 'stop_timezone', 'vehicle_type', 'platform_code']


,stop_id,stop_name,stop_lat,stop_lon,stop_desc,location_type,parent_station,wheelchair_boarding,stop_timezone,vehicle_type,platform_code
0,NSR:Quay:100140,Rennes kryss,58.020909,7.412990,NaN,NaN,NSR:StopPlace:58401,NaN,NaN,700.0,NaN
1,NSR:Quay:100141,Fjellheim snuplass,58.329847,7.598473,NaN,NaN,NSR:StopPlace:58402,NaN,NaN,700.0,NaN
2,NSR:Quay:100157,Frikstad øst,58.521609,7.925359,NaN,NaN,NSR:StopPlace:58414,NaN,NaN,700.0,NaN
3,NSR:Quay:100162,Vigeland sentrum,58.084750,7.303430,NaN,NaN,NSR:StopPlace:58419,NaN,NaN,700.0,NaN
4,NSR:Quay:100163,Vigeland sentrum,58.084771,7.303030,NaN,NaN,NSR:StopPlace:58419,NaN,NaN,700.0,NaN


routes columns:
['agency_id', 'route_id', 'route_short_name', 'route_long_name', 'route_type', 'route_desc', 'route_url', 'route_color', 'route_text_color']


,agency_id,route_id,route_short_name,route_long_name,route_type,route_desc,route_url,route_color,route_text_color
0,AKT:Authority:AKT_ID,AKT:Line:100,100,Arendal-Kristiansand,701,NaN,NaN,000000,FFFF00
1,AKT:Authority:AKT_ID,AKT:Line:101,101,Eydehavn-Arendal-Grimstad N/S,704,NaN,NaN,000000,FFFF00
2,AKT:Authority:AKT_ID,AKT:Line:1010,10,Sykehuset - Kvadraturen,704,NaN,NaN,000000,FFFF00
3,AKT:Authority:AKT_ID,AKT:Line:1012,12,Kjos haveby - Justvik,704,NaN,NaN,000000,FFFF00
4,AKT:Authority:AKT_ID,AKT:Line:1013,13,Lokalbuss Grimsmyra - Lund,704,NaN,NaN,000000,FFFF00


trips columns:
['route_id', 'trip_id', 'service_id', 'trip_headsign', 'direction_id', 'shape_id', 'wheelchair_accessible']


,route_id,trip_id,service_id,trip_headsign,direction_id,shape_id,wheelchair_accessible
0,AKT:Line:1031,AKT:ServiceJourney:11760_8222_44_117840_120120...,AKT:DayType:69_1,Kristiansand via Baneheitunnelen,1,AKT:JourneyPattern:65882_1,NaN
1,AKT:Line:1031,AKT:ServiceJourney:11761_8242_44_125040_127320...,AKT:DayType:69_1,Kristiansand via Baneheitunnelen,1,AKT:JourneyPattern:65882_1,NaN
2,AKT:Line:1012,AKT:ServiceJourney:12744_9382_45_147720_150420...,AKT:DayType:68_1,Kjos haveby,1,AKT:JourneyPattern:110503_1,NaN
3,AKT:Line:1012,AKT:ServiceJourney:12745_9402_45_151320_154020...,AKT:DayType:68_1,Kjos haveby,1,AKT:JourneyPattern:110503_1,NaN
4,AKT:Line:1012,AKT:ServiceJourney:12746_9412_45_154920_157620...,AKT:DayType:68_1,Kjos haveby,1,AKT:JourneyPattern:110503_1,NaN


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,AKT:DayType:100_1,0,1,0,1,0,0,0,20260506,20261005
1,AKT:DayType:101_1,0,0,0,1,1,0,0,20260502,20260930
2,AKT:DayType:102_1,1,1,1,0,1,0,0,20260506,20260927
3,AKT:DayType:105_1,1,0,1,1,1,0,0,20260505,20260927
4,AKT:DayType:106_1,1,1,1,1,1,1,0,20260506,20261004


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,AKT:DayType:100_1,20260514,2
1,AKT:DayType:100_1,20260623,2
2,AKT:DayType:100_1,20260625,2
3,AKT:DayType:100_1,20260630,2
4,AKT:DayType:100_1,20260702,2


## Comment

The GTFS tables contain the fields needed for the three comparison levels:

- For the **station level**, `stops.txt` includes stop IDs, names, coordinates, `location_type`, and `parent_station`. The stop IDs already use NeTEx-like identifiers such as `NSR:Quay`, while `parent_station` points to `NSR:StopPlace`

- For the **line level**, `routes.txt` contains `route_short_name`, `route_long_name`, and `route_type`, which can be used to compare public line labels

- For the **calendar level**, both `calendar.txt` and `calendar_dates.txt` are available. This means the service validity can be reconstructed using weekly rules together with added or removed exception dates

## GTFS stop hierarchy

I now inspect the stop hierarchy in the Norwegian GTFS data. This is important because the stop IDs appear to describe quays, while `parent_station` points to StopPlace IDs. The result will help decide what should be used as the GTFS station-level unit.

In [6]:
# Inspect GTFS stop hierarchy

stop_hierarchy_summary = pd.DataFrame({
    "metric": [
        "number of stop rows",
        "unique stop_id values",
        "rows with parent_station",
        "rows without parent_station",
        "unique parent_station values",
        "missing location_type values"
    ],
    "value": [
        len(stops),
        stops["stop_id"].nunique(),
        stops["parent_station"].notna().sum(),
        stops["parent_station"].isna().sum(),
        stops["parent_station"].nunique(),
        stops["location_type"].isna().sum()
    ]
})

display(stop_hierarchy_summary)

print("location_type distribution:")
display(
    stops["location_type"]
    .fillna("missing")
    .value_counts()
    .reset_index()
    .rename(columns={"index": "location_type", "location_type": "count"})
)

print("Example stop hierarchy rows:")
display(
    stops[["stop_id", "stop_name", "location_type", "parent_station", "stop_lat", "stop_lon"]]
    .head(10)
)

,metric,value
0,number of stop rows,142702
1,unique stop_id values,142702
2,rows with parent_station,91173
3,rows without parent_station,51529
4,unique parent_station values,51529
5,missing location_type values,91173


location_type distribution:


,count,count
0,missing,91173
1,1.0,51529


Example stop hierarchy rows:


,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
0,NSR:Quay:100140,Rennes kryss,NaN,NSR:StopPlace:58401,58.020909,7.412990
1,NSR:Quay:100141,Fjellheim snuplass,NaN,NSR:StopPlace:58402,58.329847,7.598473
2,NSR:Quay:100157,Frikstad øst,NaN,NSR:StopPlace:58414,58.521609,7.925359
3,NSR:Quay:100162,Vigeland sentrum,NaN,NSR:StopPlace:58419,58.084750,7.303430
4,NSR:Quay:100163,Vigeland sentrum,NaN,NSR:StopPlace:58419,58.084771,7.303030
5,NSR:Quay:100255,Landvik skole,NaN,NSR:StopPlace:23255,58.348142,8.535009
6,NSR:Quay:100275,Vrålstad,NaN,NSR:StopPlace:58574,58.839169,8.100770
7,NSR:Quay:100282,Dølemo skule,NaN,NSR:StopPlace:61230,58.711451,8.339680
8,NSR:Quay:100298,Espeskarveien kryss,NaN,NSR:StopPlace:58584,58.539808,7.107999
9,NSR:Quay:100523,Dale,NaN,NSR:StopPlace:58778,58.241107,7.881188


## Comment

The Norwegian GTFS stop table has a clear hierarchy.  

There are 142,702 stop rows in total. Most rows with `parent_station` are quay-level stops (`NSR:Quay`), while the rows without `parent_station` have `location_type = 1` and represent station-level StopPlaces.

## Build GTFS station-level table

Since the Norwegian GTFS data has a stop hierarchy, I now separate the station-level StopPlace rows from the quay-level rows.  
The goal is to check whether the `parent_station` values from quay rows match real station rows in `stops.txt`.

In [7]:
# Build GTFS station-level table

gtfs_station_rows = stops[stops["location_type"] == 1].copy()
gtfs_quay_rows = stops[stops["parent_station"].notna()].copy()

parent_station_ids = set(gtfs_quay_rows["parent_station"].dropna())
station_stop_ids = set(gtfs_station_rows["stop_id"].dropna())

station_hierarchy_check = pd.DataFrame({
    "metric": [
        "station-level rows (location_type = 1)",
        "quay rows with parent_station",
        "unique parent_station values",
        "parent_station IDs found as station stop_id",
        "parent_station IDs missing from station rows"
    ],
    "value": [
        len(gtfs_station_rows),
        len(gtfs_quay_rows),
        len(parent_station_ids),
        len(parent_station_ids & station_stop_ids),
        len(parent_station_ids - station_stop_ids)
    ]
})

display(station_hierarchy_check)

print("Example GTFS station-level rows:")
display(
    gtfs_station_rows[
        ["stop_id", "stop_name", "location_type", "parent_station", "stop_lat", "stop_lon"]
    ].head(10)
)

,metric,value
0,station-level rows (location_type = 1),51529
1,quay rows with parent_station,91173
2,unique parent_station values,51529
3,parent_station IDs found as station stop_id,51529
4,parent_station IDs missing from station rows,0


Example GTFS station-level rows:


,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
6247,NSR:StopPlace:19881,Sigurdsbu,1.0,NaN,59.677541,7.546677
6248,NSR:StopPlace:19948,Haukeli,1.0,NaN,59.735009,7.550864
6249,NSR:StopPlace:21279,Sæsvatn,1.0,NaN,59.658592,7.502200
6250,NSR:StopPlace:21367,Vamark,1.0,NaN,59.714885,7.577299
6251,NSR:StopPlace:21523,Stokketjønn,1.0,NaN,59.667051,7.529256
6252,NSR:StopPlace:21717,Tvøråsen,1.0,NaN,58.327245,6.998855
6253,NSR:StopPlace:21718,Ravnås,1.0,NaN,58.232717,7.931170
6254,NSR:StopPlace:21719,Ravnåsvegen 347,1.0,NaN,58.252313,7.950407
6255,NSR:StopPlace:21721,Hageveien,1.0,NaN,58.647459,9.107478
6256,NSR:StopPlace:21722,Reddal,1.0,NaN,58.340942,8.458131


## Comment

The GTFS stop hierarchy is consistent.

There are 51,529 station-level rows with `location_type = 1`. These rows use `NSR:StopPlace` IDs. The quay-level rows use `parent_station`, and all 51,529 unique `parent_station` values are found as real station rows in the same `stops.txt` table.

This means the GTFS station-level unit for Norway is clear: the comparison should use the `location_type = 1` StopPlace rows, not the raw quay rows.

## Prepare GTFS station-level dataset

The GTFS stop hierarchy is clear, so I now prepare a clean station-level table for later comparison.  
This table only keeps the StopPlace rows (`location_type = 1`) and the fields needed later: ID, name, latitude, and longitude.

In [8]:
# Prepare GTFS station-level dataset

gtfs_stations = gtfs_station_rows[
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

gtfs_stations = gtfs_stations.rename(columns={
    "stop_id": "gtfs_stopplace_id",
    "stop_name": "gtfs_stop_name",
    "stop_lat": "gtfs_lat",
    "stop_lon": "gtfs_lon"
})

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "station rows",
        "unique station IDs",
        "missing station names",
        "missing latitude values",
        "missing longitude values",
        "duplicate station IDs"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["gtfs_stopplace_id"].nunique(),
        gtfs_stations["gtfs_stop_name"].isna().sum(),
        gtfs_stations["gtfs_lat"].isna().sum(),
        gtfs_stations["gtfs_lon"].isna().sum(),
        gtfs_stations["gtfs_stopplace_id"].duplicated().sum()
    ]
})

display(gtfs_station_quality)

display(gtfs_stations.head(10))

,metric,value
0,station rows,51529
1,unique station IDs,51529
2,missing station names,0
3,missing latitude values,0
4,missing longitude values,0
5,duplicate station IDs,0


,gtfs_stopplace_id,gtfs_stop_name,gtfs_lat,gtfs_lon
6247,NSR:StopPlace:19881,Sigurdsbu,59.677541,7.546677
6248,NSR:StopPlace:19948,Haukeli,59.735009,7.550864
6249,NSR:StopPlace:21279,Sæsvatn,59.658592,7.502200
6250,NSR:StopPlace:21367,Vamark,59.714885,7.577299
6251,NSR:StopPlace:21523,Stokketjønn,59.667051,7.529256
6252,NSR:StopPlace:21717,Tvøråsen,58.327245,6.998855
6253,NSR:StopPlace:21718,Ravnås,58.232717,7.931170
6254,NSR:StopPlace:21719,Ravnåsvegen 347,58.252313,7.950407
6255,NSR:StopPlace:21721,Hageveien,58.647459,9.107478
6256,NSR:StopPlace:21722,Reddal,58.340942,8.458131


## Comment

The GTFS station-level dataset is ready for later comparison.

It contains `51,529 station-level StopPlace` rows. Each station has a unique ID, and there are no missing names, latitude values, longitude values, or duplicate station IDs.


## GTFS routes

In [9]:
# Inspect GTFS route-level fields

route_summary = pd.DataFrame({
    "metric": [
        "route rows",
        "unique route_id values",
        "missing route_short_name",
        "missing route_long_name",
        "unique route_short_name values",
        "unique route_type values"
    ],
    "value": [
        len(routes),
        routes["route_id"].nunique(),
        routes["route_short_name"].isna().sum(),
        routes["route_long_name"].isna().sum(),
        routes["route_short_name"].nunique(),
        routes["route_type"].nunique()
    ]
})

display(route_summary)

print("route_type distribution:")
display(
    routes["route_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "route_type", "route_type": "count"})
)

print("Example route rows:")
display(
    routes[["route_id", "route_short_name", "route_long_name", "route_type"]]
    .head(10)
)

,metric,value
0,route rows,4420
1,unique route_id values,4420
2,missing route_short_name,119
3,missing route_long_name,327
4,unique route_short_name values,2460
5,unique route_type values,28


route_type distribution:


,count,count
0,704,1957
1,712,1581
2,701,241
3,1004,150
4,1102,106
5,1014,73
6,705,63
7,1008,47
8,100,39
9,202,38


Example route rows:


,route_id,route_short_name,route_long_name,route_type
0,AKT:Line:100,100,Arendal-Kristiansand,701
1,AKT:Line:101,101,Eydehavn-Arendal-Grimstad N/S,704
2,AKT:Line:1010,10,Sykehuset - Kvadraturen,704
3,AKT:Line:1012,12,Kjos haveby - Justvik,704
4,AKT:Line:1013,13,Lokalbuss Grimsmyra - Lund,704
5,AKT:Line:1015,15,Tinnheia - Lund - UiA,704
6,AKT:Line:1019,19,Suldalen - Gimlekollen,704
7,AKT:Line:102,102,Tromøy Øst-Arendal-Rykene,704
8,AKT:Line:103,103,Tromøy Vest-Arendal-Hisøy,704
9,AKT:Line:1030,30,Kvadraturen - Mosby - Vennesl,704


## Comment

The GTFS route table contains 4,420 route rows, and each route has a unique `route_id`.

Most routes have a public `route_short_name`, but 119 routes are missing this value. There are 2,460 unique `route_short_name` values, which means that several route rows share the same public line label.

The route table also contains many different `route_type` values. This shows that the Norwegian GTFS dataset includes several transport modes, not only one type of service.

For the later line-level comparison, `route_short_name` is still the most useful public-facing field, but it will likely need deduplication before comparing it with NeTEx.

## GTFS transport modes

The Norwegian GTFS dataset uses many different `route_type` values.  
I now group these codes into readable transport modes to understand which types of transport are included in the dataset.

In [10]:
# Create readable transport mode table from route_type
def classify_route_type(route_type):
    if 100 <= route_type < 200:
        return "Rail"
    elif 200 <= route_type < 300:
        return "Coach"
    elif 300 <= route_type < 400:
        return "Suburban rail"
    elif 400 <= route_type < 500:
        return "Metro / urban rail"
    elif 500 <= route_type < 600:
        return "Underground / Metro"
    elif 600 <= route_type < 700:
        return "Ferry"
    elif 700 <= route_type < 800:
        return "Bus"
    elif 800 <= route_type < 900:
        return "Trolleybus"
    elif 900 <= route_type < 1000:
        return "Tram"
    elif 1000 <= route_type < 1100:
        return "Ferry / water transport"
    elif 1100 <= route_type < 1200:
        return "Air"
    elif 1300 <= route_type < 1400:
        return "Cable / aerial transport"
    else:
        route_type_basic = {
            0: "Tram",
            1: "Metro",
            2: "Rail",
            3: "Bus",
            4: "Ferry",
            7: "Funicular",
            11: "Trolleybus",
            12: "Monorail"
        }
        return route_type_basic.get(route_type, f"Unknown ({route_type})")

gtfs_route_modes = routes.copy()
gtfs_route_modes["route_type"] = gtfs_route_modes["route_type"].astype(int)
gtfs_route_modes["transport_mode"] = gtfs_route_modes["route_type"].apply(classify_route_type)

transport_mode_table = (
    gtfs_route_modes
    .groupby(["route_type", "transport_mode"])
    .size()
    .reset_index(name="number_of_routes")
    .sort_values("number_of_routes", ascending=False)
)

display(transport_mode_table)

,route_type,transport_mode,number_of_routes
12,704,Bus,1957
16,712,Bus,1581
10,701,Bus,241
21,1004,Ferry / water transport,150
26,1102,Air,106
24,1014,Ferry / water transport,73
13,705,Bus,63
22,1008,Ferry / water transport,47
0,100,Rail,39
7,202,Coach,38


## Comment

The Norwegian GTFS dataset is clearly multimodal.

Most route rows are bus services, especially route types 704 and 712. The dataset also includes ferry or water transport, air services, rail, coach, tram, metro / urban rail, and one cable or aerial transport route.

## Prepare GTFS line-level dataset

The GTFS route table contains technical route rows, but several rows can share the same visible line label.  
For the later line-level comparison, I therefore prepare a simpler table based on `route_short_name`.

The code first removes routes without a `route_short_name`, because these routes do not have a public line label that can be compared easily.  
Then it counts how many route rows exist, how many have or do not have a line label, and how many unique line labels are present.

After that, the routes are grouped by `route_short_name`.  
This creates one row per public line label and keeps useful information such as the number of technical route rows behind the label, the route type codes, and one example long name.

In [11]:
# Prepare GTFS line-level dataset

gtfs_routes_with_label = routes[routes["route_short_name"].notna()].copy()

gtfs_line_summary = pd.DataFrame({
    "metric": [
        "route rows",
        "route rows with route_short_name",
        "route rows without route_short_name",
        "unique route_short_name values",
        "duplicate route_short_name rows"
    ],
    "value": [
        len(routes),
        len(gtfs_routes_with_label),
        routes["route_short_name"].isna().sum(),
        gtfs_routes_with_label["route_short_name"].nunique(),
        gtfs_routes_with_label["route_short_name"].duplicated().sum()
    ]
})

display(gtfs_line_summary)

# Deduplicated GTFS line-level table
gtfs_lines = (
    gtfs_routes_with_label
    .groupby("route_short_name", as_index=False)
    .agg(
        number_of_route_rows=("route_id", "count"),
        route_types=("route_type", lambda x: sorted(x.dropna().unique().tolist())),
        example_route_long_name=("route_long_name", "first")
    )
    .rename(columns={"route_short_name": "gtfs_line_label"})
    .sort_values("gtfs_line_label")
)

display(gtfs_lines.head(20))

,metric,value
0,route rows,4420
1,route rows with route_short_name,4301
2,route rows without route_short_name,119
3,unique route_short_name values,2460
4,duplicate route_short_name rows,1841


,gtfs_line_label,number_of_route_rows,route_types,example_route_long_name
0,001,1,[1008],Hyke Arendalsuka
1,01,1,[704],Horten-Tønsberg-Sandefjord
2,011,1,[704],Larvik-Tønsberg
3,02,4,"[704, 712]",Horten vgs
4,021,1,[704],Tønsberg-Skoppum-Holmestrand
5,022,3,"[704, 712]",Greveskogen-Tønsberg-V.Ende
6,023,1,[704],Nykirke-Horten-Tønsberg
7,1,11,"[401, 701, 704, 901]",Ranheim - Strindheim - sentrum - Tiller - Heim...
8,10,9,"[701, 704]",Sykehuset - Kvadraturen
9,100,18,"[701, 702, 704, 712]",Arendal-Kristiansand


## Comment

The GTFS line-level table is prepared at the public line-label level.

The original route table contains 4,420 route rows. Out of these, 4,301 rows have a `route_short_name`, while 119 rows do not have a public line label and are therefore not used in this line-label table.

After grouping by `route_short_name`, there are 2,460 unique public line labels. This means that many public labels appear in more than one technical route row. For example, one visible line label can be connected to several `route_id` values or several `route_type` codes.

This confirms that the later line-level comparison should not be based directly on raw route rows. Instead, the practical GTFS line-level unit should be the deduplicated public label, using `route_short_name`.

## Inspect routes without public line labels

Some GTFS route rows do not have a `route_short_name`.  
Before leaving the line-level exploration, I inspect these rows separately to understand whether they belong to specific transport modes or special services.

They are not used in the main line-label table, but checking them helps avoid ignoring part of the dataset without explanation.

In [12]:
# Inspect routes without route_short_name

routes_without_short_name = routes[routes["route_short_name"].isna()].copy()
routes_without_short_name["route_type"] = routes_without_short_name["route_type"].astype(int)
routes_without_short_name["transport_mode"] = routes_without_short_name["route_type"].apply(classify_route_type)

missing_label_summary = (
    routes_without_short_name
    .groupby(["route_type", "transport_mode"])
    .size()
    .reset_index(name="number_of_routes")
    .sort_values("number_of_routes", ascending=False)
)

display(missing_label_summary)

display(
    routes_without_short_name[
        ["route_id", "route_short_name", "route_long_name", "route_type", "transport_mode"]
    ].head(20)
)

,route_type,transport_mode,number_of_routes
8,1102,Air,106
4,711,Bus,4
1,106,Rail,3
0,105,Rail,1
2,704,Bus,1
3,710,Bus,1
5,714,Bus,1
6,1008,Ferry / water transport,1
7,1015,Ferry / water transport,1


,route_id,route_short_name,route_long_name,route_type,transport_mode
624,AVI:Line:DX_EVE-OLA,NaN,Harstad/Narvik-Ørland,1102,Air
625,AVI:Line:DX_OSL-FRO,NaN,Oslo-Florø,1102,Air
626,AVI:Line:DX_OSL-OLA,NaN,Oslo-Ørland,1102,Air
627,AVI:Line:DX_OSL-RRS,NaN,Oslo-Røros,1102,Air
628,AVI:Line:DX_OSL-SRP,NaN,Oslo-Stord,1102,Air
629,AVI:Line:DY_BGO-EVE,NaN,Bergen-Harstad/Narvik,1102,Air
630,AVI:Line:DY_BGO-TRD,NaN,Bergen-Trondheim,1102,Air
631,AVI:Line:DY_OSL-AES,NaN,Oslo-Ålesund,1102,Air
632,AVI:Line:DY_OSL-ALF,NaN,Oslo-Alta,1102,Air
633,AVI:Line:DY_OSL-ANX,NaN,Oslo-Andenes,1102,Air


## Comment

The routes without `route_short_name` were inspected separately.

Most of these missing-label routes are air services. This explains why they do not follow the same public line-label structure as many bus or rail routes. A few missing labels also appear for bus, rail, and ferry or water transport routes.

These routes are not used in the main deduplicated line-label table because they do not have a clear `route_short_name` for label-based comparison. 

## GTFS calendar / validity level


In [13]:
# Inspect GTFS calendar structure

calendar_work = calendar.copy()
calendar_dates_work = calendar_dates.copy()

# Convert GTFS dates to datetime
calendar_work["start_date"] = pd.to_datetime(calendar_work["start_date"].astype(str), format="%Y%m%d")
calendar_work["end_date"] = pd.to_datetime(calendar_work["end_date"].astype(str), format="%Y%m%d")
calendar_dates_work["date"] = pd.to_datetime(calendar_dates_work["date"].astype(str), format="%Y%m%d")

weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

calendar_summary = pd.DataFrame({
    "metric": [
        "calendar rows",
        "unique service_id values in calendar",
        "earliest start_date",
        "latest end_date",
        "calendar_dates rows",
        "unique service_id values in calendar_dates",
        "earliest exception date",
        "latest exception date"
    ],
    "value": [
        len(calendar_work),
        calendar_work["service_id"].nunique(),
        calendar_work["start_date"].min(),
        calendar_work["end_date"].max(),
        len(calendar_dates_work),
        calendar_dates_work["service_id"].nunique(),
        calendar_dates_work["date"].min(),
        calendar_dates_work["date"].max()
    ]
})

display(calendar_summary)

print("Exception type distribution:")
display(
    calendar_dates_work["exception_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "exception_type", "exception_type": "count"})
)

print("Example calendar rows:")
display(calendar_work.head(10))

print("Example calendar_dates rows:")
display(calendar_dates_work.head(10))

,metric,value
0,calendar rows,3798
1,unique service_id values in calendar,3798
2,earliest start_date,2019-04-13 00:00:00
3,latest end_date,2027-03-31 00:00:00
4,calendar_dates rows,290234
5,unique service_id values in calendar_dates,7826
6,earliest exception date,2019-04-19 00:00:00
7,latest exception date,2027-06-20 00:00:00


Exception type distribution:


,count,count
0,1,264550
1,2,25684


Example calendar rows:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,AKT:DayType:100_1,0,1,0,1,0,0,0,2026-05-06,2026-10-05
1,AKT:DayType:101_1,0,0,0,1,1,0,0,2026-05-02,2026-09-30
2,AKT:DayType:102_1,1,1,1,0,1,0,0,2026-05-06,2026-09-27
3,AKT:DayType:105_1,1,0,1,1,1,0,0,2026-05-05,2026-09-27
4,AKT:DayType:106_1,1,1,1,1,1,1,0,2026-05-06,2026-10-04
5,AKT:DayType:107_1,0,0,0,0,1,0,0,2026-05-09,2026-10-08
6,AKT:DayType:108_1,1,1,1,1,1,0,0,2026-08-13,2026-10-04
7,AKT:DayType:109_1,1,1,1,1,1,0,0,2026-08-13,2026-09-27
8,AKT:DayType:10_1,0,1,1,1,1,0,0,2026-05-06,2026-09-28
9,AKT:DayType:110_1,0,0,1,0,0,0,0,2026-08-13,2026-09-29


Example calendar_dates rows:


,service_id,date,exception_type
0,AKT:DayType:100_1,2026-05-14,2
1,AKT:DayType:100_1,2026-06-23,2
2,AKT:DayType:100_1,2026-06-25,2
3,AKT:DayType:100_1,2026-06-30,2
4,AKT:DayType:100_1,2026-07-02,2
5,AKT:DayType:100_1,2026-07-07,2
6,AKT:DayType:100_1,2026-07-09,2
7,AKT:DayType:100_1,2026-07-14,2
8,AKT:DayType:100_1,2026-07-16,2
9,AKT:DayType:100_1,2026-07-21,2


## Comment

The GTFS calendar structure is available and can be used for the validity analysis.

The `calendar.txt` table contains 3,798 service IDs with weekly operating rules and validity periods. The date range is quite broad, from 2019 to 2027.

The `calendar_dates.txt` table is much larger, with 290,234 exception rows. It contains both added service days (`exception_type = 1`) and removed service days (`exception_type = 2`). Most exceptions are added service days.

This means the calendar comparison cannot rely only on the basic weekly rules from `calendar.txt`. The exceptions in `calendar_dates.txt` are important and must be included when rebuilding the real active service days.

## Check GTFS service_id coverage

Before rebuilding the active service days, I check how `service_id` values are shared between `trips.txt`, `calendar.txt`, and `calendar_dates.txt`.

This is important because the calendar logic depends on both weekly rules and exceptions. Some service IDs may appear in `calendar_dates.txt` but not in `calendar.txt`, so they need to be understood before creating the final validity patterns.

In [14]:
# Check service_id coverage across GTFS tables

trip_service_ids = set(trips["service_id"].dropna())
calendar_service_ids = set(calendar_work["service_id"].dropna())
calendar_dates_service_ids = set(calendar_dates_work["service_id"].dropna())

service_id_coverage = pd.DataFrame({
    "metric": [
        "unique service_id values in trips",
        "unique service_id values in calendar",
        "unique service_id values in calendar_dates",
        "trip service_ids found in calendar",
        "trip service_ids found in calendar_dates",
        "trip service_ids missing from both calendar and calendar_dates",
        "calendar_dates service_ids not in calendar",
        "calendar service_ids not used in trips"
    ],
    "value": [
        len(trip_service_ids),
        len(calendar_service_ids),
        len(calendar_dates_service_ids),
        len(trip_service_ids & calendar_service_ids),
        len(trip_service_ids & calendar_dates_service_ids),
        len(trip_service_ids - calendar_service_ids - calendar_dates_service_ids),
        len(calendar_dates_service_ids - calendar_service_ids),
        len(calendar_service_ids - trip_service_ids)
    ]
})

display(service_id_coverage)

print("Example service_ids in calendar_dates but not in calendar:")
display(
    calendar_dates_work[
        calendar_dates_work["service_id"].isin(calendar_dates_service_ids - calendar_service_ids)
    ].head(10)
)

,metric,value
0,unique service_id values in trips,9854
1,unique service_id values in calendar,3798
2,unique service_id values in calendar_dates,7826
3,trip service_ids found in calendar,3798
4,trip service_ids found in calendar_dates,7826
5,trip service_ids missing from both calendar an...,0
6,calendar_dates service_ids not in calendar,6056
7,calendar service_ids not used in trips,0


Example service_ids in calendar_dates but not in calendar:


,service_id,date,exception_type
68,AKT:DayType:103_1,2026-05-15,1
69,AKT:DayType:103_1,2026-08-13,1
70,AKT:DayType:103_1,2026-08-14,1
71,AKT:DayType:103_1,2026-09-28,1
72,AKT:DayType:103_1,2026-09-29,1
73,AKT:DayType:103_1,2026-10-01,1
74,AKT:DayType:103_1,2026-10-02,1
75,AKT:DayType:104_1,2026-05-15,1
76,AKT:DayType:104_1,2026-08-13,1
77,AKT:DayType:104_1,2026-08-14,1


## Comment

The service ID coverage is consistent.

There are 9,854 service IDs used by trips. All of them are found either in `calendar.txt` or in `calendar_dates.txt`, so no trip service is missing validity information.

However, 6,056 service IDs appear in `calendar_dates.txt` but not in `calendar.txt`. This means that many services are defined only through specific exception dates, not through regular weekly calendar rules.

For the calendar reconstruction, both cases must be handled: services with weekly rules from `calendar.txt`, and exception-only services from `calendar_dates.txt`.

## Rebuild GTFS active service days

The GTFS calendar information is split across `calendar.txt` and `calendar_dates.txt`.

For services in `calendar.txt`, the code first creates active dates from the weekly rules between `start_date` and `end_date`.  
Then it applies the exceptions from `calendar_dates.txt`: added dates are included, and removed dates are excluded.

Some services only appear in `calendar_dates.txt`. For these services, the active dates come only from the added exception dates.

In [15]:
# Make sure exception_type is numeric
calendar_dates_work["exception_type"] = calendar_dates_work["exception_type"].astype(int)

# Separate added and removed dates
added_dates = defaultdict(set)
removed_dates = defaultdict(set)

for _, row in calendar_dates_work.iterrows():
    if row["exception_type"] == 1:
        added_dates[row["service_id"]].add(row["date"])
    elif row["exception_type"] == 2:
        removed_dates[row["service_id"]].add(row["date"])

# Rebuild active dates for each service_id
gtfs_active_dates = {}

# 1. Services with regular weekly calendar rules
for _, row in calendar_work.iterrows():
    service_id = row["service_id"]
    start = row["start_date"]
    end = row["end_date"]
    
    all_dates = pd.date_range(start, end, freq="D")
    active = set()
    
    for d in all_dates:
        weekday_col = weekday_cols[d.weekday()]
        if row[weekday_col] == 1:
            active.add(d)
    
    # Apply exceptions
    active = (active | added_dates[service_id]) - removed_dates[service_id]
    gtfs_active_dates[service_id] = active

# 2. Services only defined through calendar_dates
for service_id in calendar_dates_service_ids - calendar_service_ids:
    active = added_dates[service_id] - removed_dates[service_id]
    gtfs_active_dates[service_id] = active

# Create summary table
gtfs_calendar_summary = []

for service_id, active in gtfs_active_dates.items():
    gtfs_calendar_summary.append({
        "service_id": service_id,
        "n_active_days": len(active),
        "first_active_date": min(active) if active else pd.NaT,
        "last_active_date": max(active) if active else pd.NaT,
        "calendar_source": (
            "calendar + exceptions" if service_id in calendar_service_ids else "calendar_dates only"
        )
    })

gtfs_calendar_summary = pd.DataFrame(gtfs_calendar_summary)

display(gtfs_calendar_summary.head(20))

display(
    gtfs_calendar_summary["calendar_source"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "calendar_source", "calendar_source": "number_of_services"})
)

,service_id,n_active_days,first_active_date,last_active_date,calendar_source
0,AKT:DayType:100_1,27,2026-05-07,2026-10-01,calendar + exceptions
1,AKT:DayType:101_1,24,2026-05-07,2026-09-25,calendar + exceptions
2,AKT:DayType:102_1,48,2026-05-06,2026-09-25,calendar + exceptions
3,AKT:DayType:105_1,48,2026-05-06,2026-09-25,calendar + exceptions
4,AKT:DayType:106_1,83,2026-05-06,2026-10-03,calendar + exceptions
5,AKT:DayType:107_1,9,2026-05-15,2026-10-02,calendar + exceptions
6,AKT:DayType:108_1,37,2026-08-13,2026-10-02,calendar + exceptions
7,AKT:DayType:109_1,32,2026-08-13,2026-09-25,calendar + exceptions
8,AKT:DayType:10_1,51,2026-05-06,2026-09-25,calendar + exceptions
9,AKT:DayType:110_1,6,2026-08-19,2026-09-23,calendar + exceptions


,number_of_services,count
0,calendar_dates only,6056
1,calendar + exceptions,3798


## Output

The GTFS active service days were rebuilt successfully.

The result now combines the regular weekly rules from `calendar.txt` with the added and removed dates from `calendar_dates.txt`. In total, there are two types of services: 3,798 services use regular calendar rules plus exceptions, while 6,056 services are defined only through `calendar_dates.txt`.

# Part 2: Norway NeTEx Exploration

In [16]:
# Set NeTEx path

netex_zip_path = data_dir / "rb_norway-aggregated-netex.zip"

print("NeTEx path:")
print(netex_zip_path)

print("\nNeTEx file exists:")
print(netex_zip_path.exists())

NeTEx path:
data/rb_norway-aggregated-netex.zip

NeTEx file exists:
True


In [17]:
# Inspect NeTEx ZIP contents
with zipfile.ZipFile(netex_zip_path, "r") as z:
    netex_files = pd.DataFrame([
        {
            "name": info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(netex_files.sort_values("size_mb", ascending=False).head(30))

print("Number of files in NeTEx ZIP:", len(netex_files))

,name,size_mb
4454,_stops.xml,403.53
1898,_VYX_shared_data.xml,70.57
1426,_INN_shared_data.xml,68.54
1715,VKT_VKT-Line-2_02_Tjome-Tonsberg-Horten-Holmes...,66.93
1287,KOL_KOL-Line-8_1002_2_2.xml,55.52
969,SKY_SKY-Line-1_1_Bergen-lufthavn-Flesland--Lag...,45.83
1691,SKY_SKY-Line-4_4_Flaktveit---Hesjaholtet.xml,43.82
4740,FLT_FLT-Line-FLY1_FLY1_FLY1.xml,42.67
1698,RUT_RUT-Line-3_3_Kolsas---Mortensrud.xml,40.74
2754,_SOF_shared_data.xml,40.23


Number of files in NeTEx ZIP: 5024


## Comment

The Norway NeTEx ZIP contains 5,024 XML files, so the data is split across many files.

The largest file is `_stops.xml`, with about 403 MB. This probably contains the stop or StopPlace information. There are also several shared data files and many line-specific XML files.

Because the NeTEx dataset is large, especially the stops file, it is better to inspect the structure first and avoid loading everything blindly into memory.

## Inspect Netex XML structure

Before extracting data from NeTEx, I check which important NeTEx elements appear in selected XML files.

The goal is to see where elements such as `StopPlace`, `Quay`, `Line`, `ServiceJourney`, `DayType`, and `UicOperatingPeriod` appear before deciding the full parsing strategy.

In [18]:
# Helper to remove XML namespace
def local_name(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

# Select a small set of files to inspect
netex_file_names = netex_files["name"].tolist()

stops_files = [f for f in netex_file_names if f.endswith("_stops.xml") or f.endswith("stops.xml")]
shared_files = [f for f in netex_file_names if "shared_data" in f]
line_files = [f for f in netex_file_names if "-Line-" in f]

sample_netex_files = (
    stops_files[:1] +
    shared_files[:3] +
    line_files[:5]
)

sample_netex_files

['_stops.xml',
 '_BOR_flexible_shared_data.xml',
 '_AKT_shared_data.xml',
 '_VYX_flexible_shared_data.xml',
 'VKT_VKT-Line-1213_1213_Bugarden-Sagmyra.xml',
 'INN_INN-Line-312_312_Lidar---Oystre-Slidre.xml',
 'OST_OST-Line-1_472_472_Skoler-Rakkestad.xml',
 'KOL_KOL-Line-8_2643_SK2201_SK2201.xml',
 'SKY_SKY-Line-460E_460E_Agotnes---Bergen.xml']

In [19]:
# Target NeTEx elements for the MVD levels
target_tags = {
    "StopPlace",
    "Quay",
    "Line",
    "Route",
    "ServiceJourney",
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "UicOperatingPeriod"
}

def scan_xml_elements_in_zip(zip_path, member_name, target_tags):
    counts = Counter()
    
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                
                if tag in target_tags:
                    counts[tag] += 1
                
                # clear element to save memory
                elem.clear()
    
    return counts

structure_rows = []

for file_name in sample_netex_files:
    counts = scan_xml_elements_in_zip(netex_zip_path, file_name, target_tags)
    
    row = {"file": file_name}
    for tag in sorted(target_tags):
        row[tag] = counts.get(tag, 0)
    
    structure_rows.append(row)

netex_structure_check = pd.DataFrame(structure_rows)

display(netex_structure_check)

,file,DayType,DayTypeAssignment,Line,OperatingPeriod,Quay,Route,ServiceJourney,StopPlace,UicOperatingPeriod
0,_stops.xml,0,0,0,0,101759,0,0,57957,0
1,_BOR_flexible_shared_data.xml,5,18,0,18,0,0,0,0,0
2,_AKT_shared_data.xml,133,1834,0,99,0,0,0,0,0
3,_VYX_flexible_shared_data.xml,1,1,0,1,0,0,0,0,0
4,VKT_VKT-Line-1213_1213_Bugarden-Sagmyra.xml,0,0,1,0,0,2,4,0,0
5,INN_INN-Line-312_312_Lidar---Oystre-Slidre.xml,0,0,1,0,0,2,22,0,0
6,OST_OST-Line-1_472_472_Skoler-Rakkestad.xml,0,0,1,0,0,4,4,0,0
7,KOL_KOL-Line-8_2643_SK2201_SK2201.xml,0,0,1,0,0,3,4,0,0
8,SKY_SKY-Line-460E_460E_Agotnes---Bergen.xml,0,0,1,0,0,2,105,0,0


## Output

The NeTEx ZIP contains multiple XML files, each with a different purpose:

- **`_stops.xml`** — contains all stops and stations for Norway: 57,957 `StopPlace` and 101,759 `Quay` elements

- **`_shared_data.xml` files** (e.g. `_AKT_shared_data.xml`, `_BOR_shared_data.xml`) — contain calendar data per operator: `DayType`, `DayTypeAssignment` and `OperatingPeriod` elements. Each operator has its own shared data file

- **Line-specific files** (e.g. `VKT_VKT-Line-1213...xml`) — contain one `Line`, its `Route` and `ServiceJourney` elements

This is a well-structured feed that separates stops, calendar and timetable data into different files.

## Inspecting a Sample of NeTEx StopPlace Elements

Before extracting all stations from the NeTEx file, I first look at a small sample of 10 `StopPlace` elements to understand the structure and check that the key fields like ID, name, latitude and longitude are present and readable. Since the stops file can be large, I use a memory-efficient streaming approach that reads the file element by element and stops as soon as 10 stops have been collected.

In [20]:
# Helper function that first looks for a direct child with that tag, then falls back to any descendant (handles both flat and nested structures) 

def get_first_text_by_tag(elem, tag):
    ns_url = "http://www.netex.org.uk/netex"
    child = elem.find(f"{{{ns_url}}}{tag}")
    if child is None:
        child = elem.find(f".//{{{ns_url}}}{tag}")
    return child.text.strip() if child is not None and child.text else None

In [21]:
# Extract a small sample of StopPlace elements from the stops file
def extract_stopplace_sample(zip_path, member_name, max_items=10):
    rows = []

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                if local_name(elem.tag) == "StopPlace":
                    rows.append({
                        "stopplace_id": elem.attrib.get("id"),
                        "name": get_first_text_by_tag(elem, "Name"),
                        "latitude": get_first_text_by_tag(elem, "Latitude"),
                        "longitude": get_first_text_by_tag(elem, "Longitude"),
                    })

                    elem.clear()

                    if len(rows) >= max_items:
                        break

    return pd.DataFrame(rows)

netex_stopplace_sample = extract_stopplace_sample(
    netex_zip_path,
    "_stops.xml",
    max_items=10
)

display(netex_stopplace_sample)

,stopplace_id,name,latitude,longitude
0,NSR:StopPlace:1,Drangedal stasjon,59.096262,9.064246
1,NSR:StopPlace:58878,Drangedal stasjon,59.096342,9.064179
2,NSR:StopPlace:1000,Sentvedt,59.668470,11.379885
3,NSR:StopPlace:10000,Grøtholm,60.868371,11.082162
4,NSR:StopPlace:10001,Venstad vegkryss,60.689267,11.810896
5,NSR:StopPlace:10002,Husum,60.872022,11.083912
6,NSR:StopPlace:10003,Nybråtenvegen,60.295777,12.319952
7,NSR:StopPlace:10004,Skårsveen,60.877488,11.082631
8,NSR:StopPlace:10005,Bergsland,60.285184,12.357268
9,NSR:StopPlace:10006,Sleperud,60.884120,11.078382


## Comment

The StopPlace sample was extracted successfully.

The output shows that the NeTEx `StopPlace` elements contain the fields needed for the station-level analysis: ID, name, latitude, and longitude. The IDs also use the same `NSR:StopPlace` structure that was already visible in the GTFS `parent_station` and station-level rows.


## Extract all NeTEx StopPlaces

The StopPlace sample showed that the needed fields are available.  
I now extract all `StopPlace` elements from `_stops.xml` and prepare the NeTEx station-level table.

In [22]:
# Extract all StopPlace elements from _stops.xml
def extract_all_stopplaces(zip_path, member_name):
    rows = []
    current = None
    depth = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)

                if event == "start" and tag == "StopPlace":
                    depth += 1
                    if depth == 1:
                        current = {
                            "netex_stopplace_id":   elem.attrib.get("id"),
                            "netex_stopplace_name": None,
                            "netex_lat":            None,
                            "netex_lon":            None
                        }

                elif event == "end" and tag == "StopPlace":
                    if depth == 1 and current is not None:
                        rows.append(current)
                        current = None
                    depth -= 1
                    elem.clear()

                elif event == "end" and depth > 0 and current is not None:
                    text = elem.text.strip() if elem.text else None
                    if tag == "Name" and current["netex_stopplace_name"] is None:
                        current["netex_stopplace_name"] = text
                    elif tag == "Latitude" and current["netex_lat"] is None:
                        current["netex_lat"] = text
                    elif tag == "Longitude" and current["netex_lon"] is None:
                        current["netex_lon"] = text
                    elem.clear()

                elif event == "end":
                    elem.clear()

    return pd.DataFrame(rows)

netex_stopplaces = extract_all_stopplaces(
    netex_zip_path,
    "_stops.xml"
)

# Convert coordinates to numeric values
netex_stopplaces["netex_lat"] = pd.to_numeric(netex_stopplaces["netex_lat"], errors="coerce")
netex_stopplaces["netex_lon"] = pd.to_numeric(netex_stopplaces["netex_lon"], errors="coerce")

display(netex_stopplaces.head(10))

,netex_stopplace_id,netex_stopplace_name,netex_lat,netex_lon
0,NSR:StopPlace:1,Drangedal stasjon,59.096262,9.064246
1,NSR:StopPlace:58878,Drangedal stasjon,59.096342,9.064179
2,NSR:StopPlace:1000,Sentvedt,59.668470,11.379885
3,NSR:StopPlace:10000,Grøtholm,60.868371,11.082162
4,NSR:StopPlace:10001,Venstad vegkryss,60.689267,11.810896
5,NSR:StopPlace:10002,Husum,60.872022,11.083912
6,NSR:StopPlace:10003,Nybråtenvegen,60.295777,12.319952
7,NSR:StopPlace:10004,Skårsveen,60.877488,11.082631
8,NSR:StopPlace:10005,Bergsland,60.285184,12.357268
9,NSR:StopPlace:10006,Sleperud,60.884120,11.078382


In [23]:
netex_stopplace_quality = pd.DataFrame({
    "metric": [
        "StopPlace rows",
        "unique StopPlace IDs",
        "missing names",
        "missing latitude values",
        "missing longitude values",
        "duplicate StopPlace IDs"
    ],
    "value": [
        len(netex_stopplaces),
        netex_stopplaces["netex_stopplace_id"].nunique(),
        netex_stopplaces["netex_stopplace_name"].isna().sum(),
        netex_stopplaces["netex_lat"].isna().sum(),
        netex_stopplaces["netex_lon"].isna().sum(),
        netex_stopplaces["netex_stopplace_id"].duplicated().sum()
    ]
})

display(netex_stopplace_quality)

,metric,value
0,StopPlace rows,57957
1,unique StopPlace IDs,57957
2,missing names,0
3,missing latitude values,0
4,missing longitude values,0
5,duplicate StopPlace IDs,0


## Comment

The full StopPlace extraction worked well.

The `_stops.xml` file contains 57,957 `StopPlace` rows. Each StopPlace has a unique ID, and there are no missing names, latitude values, longitude values, or duplicate StopPlace IDs.

This means the NeTEx station-level dataset is clean and complete. 

## Inspect NeTEx line structure

After extracting the NeTEx StopPlaces, I now inspect the line-specific XML files.

The goal is to check which public line fields are available in NeTEx, especially `Line` ID, name, short name, public code, and transport mode.  
I also count `Route` and `ServiceJourney` elements as structural context, but the main line-level comparison later will probably use the public line information from `Line`.

In [24]:
# Select line-specific NeTEx files

line_files = [f for f in netex_files["name"].tolist() if "-Line-" in f]

print("Number of line-specific NeTEx files:", len(line_files))
print("Example line files:")
display(pd.DataFrame({"file": line_files[:10]}))

Number of line-specific NeTEx files: 4421
Example line files:


,file
0,VKT_VKT-Line-1213_1213_Bugarden-Sagmyra.xml
1,INN_INN-Line-312_312_Lidar---Oystre-Slidre.xml
2,OST_OST-Line-1_472_472_Skoler-Rakkestad.xml
3,KOL_KOL-Line-8_2643_SK2201_SK2201.xml
4,SKY_SKY-Line-460E_460E_Agotnes---Bergen.xml
5,RUT_RUT-Line-5069_506X_Sommerbuss-i-Drobak.xml
6,OST_OST-Line-1_197_197_Hvaler-barne-og-ungdoms...
7,SOF_SOF-Line-6128_69_257_Sandvik-Eikefjord.xml
8,SKY_SKY-Line-920_920_Dale---Vaksdal---Arna---B...
9,KOL_KOL-Line-8_1018_34_34.xml


This cell extracts a small sample from the NeTEx line-specific XML files.

For each sampled file, it reads the main `Line` element and extracts the line ID, name, short name, public code, and transport mode. It also counts how many `Route` and `ServiceJourney` elements appear in the same file.

In [25]:
def extract_line_sample(zip_path, file_list, max_files=20):
    rows = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for member_name in file_list[:max_files]:
            line_row = {
                "file":                   member_name,
                "line_id":                None,
                "line_name":              None,
                "short_name":             None,
                "public_code":            None,
                "transport_mode":         None,
                "route_count":            0,
                "service_journey_count":  0
            }
            current_line = None
            line_depth = 0

            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "Line":
                        line_depth += 1
                        if line_depth == 1:
                            current_line = {
                                "line_id":        elem.attrib.get("id"),
                                "line_name":      None,
                                "short_name":     None,
                                "public_code":    None,
                                "transport_mode": None
                            }

                    elif event == "end" and tag == "Line":
                        if line_depth == 1 and current_line is not None:
                            line_row.update(current_line)
                            current_line = None
                        line_depth -= 1
                        elem.clear()

                    elif event == "end" and current_line is not None and line_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "Name" and current_line["line_name"] is None:
                            current_line["line_name"] = text
                        elif tag == "ShortName" and current_line["short_name"] is None:
                            current_line["short_name"] = text
                        elif tag == "PublicCode" and current_line["public_code"] is None:
                            current_line["public_code"] = text
                        elif tag == "TransportMode" and current_line["transport_mode"] is None:
                            current_line["transport_mode"] = text
                        elem.clear()

                    elif event == "end":
                        if tag == "Route":
                            line_row["route_count"] += 1
                        elif tag == "ServiceJourney":
                            line_row["service_journey_count"] += 1
                        elem.clear()

            rows.append(line_row)

    return pd.DataFrame(rows)

netex_line_sample = extract_line_sample(
    netex_zip_path,
    line_files,
    max_files=20
)
display(netex_line_sample)

,file,line_id,line_name,short_name,public_code,transport_mode,route_count,service_journey_count
0,VKT_VKT-Line-1213_1213_Bugarden-Sagmyra.xml,VKT:Line:1213,Bugården-Sagmyra,None,1213,bus,2,4
1,INN_INN-Line-312_312_Lidar---Oystre-Slidre.xml,INN:Line:312,Lidar - Øystre Slidre,None,312,bus,2,22
2,OST_OST-Line-1_472_472_Skoler-Rakkestad.xml,OST:Line:1_472,Skoler Rakkestad,None,472,bus,4,4
3,KOL_KOL-Line-8_2643_SK2201_SK2201.xml,KOL:Line:8_2643,SK2201,None,SK2201,bus,3,4
4,SKY_SKY-Line-460E_460E_Agotnes---Bergen.xml,SKY:Line:460E,Ågotnes - Bergen,None,460E,bus,2,105
5,RUT_RUT-Line-5069_506X_Sommerbuss-i-Drobak.xml,RUT:Line:5069,Sommerbuss i Drøbak,None,506X,bus,2,44
6,OST_OST-Line-1_197_197_Hvaler-barne-og-ungdoms...,OST:Line:1_197,Hvaler-barne og ungdomsskole,None,197,bus,12,24
7,SOF_SOF-Line-6128_69_257_Sandvik-Eikefjord.xml,SOF:Line:6128_69,Sandvik-Eikefjord,None,257,bus,6,7
8,SKY_SKY-Line-920_920_Dale---Vaksdal---Arna---B...,SKY:Line:920,Dale - Vaksdal - Arna - Bergen,None,920,bus,8,14
9,KOL_KOL-Line-8_1018_34_34.xml,KOL:Line:8_1018,34,None,34,bus,15,58


## Comment

There are 4,421 line-specific XML files. The sample shows that these files contain usable line information, especially `line_id`, `line_name`, `public_code`, and `transport_mode`. The `short_name` field is empty in this sample, so `public_code` looks like the better NeTEx field for the later line-label comparison.

The files also contain `Route` and `ServiceJourney` elements. These are useful as structural context, but the main line-level comparison will probably be based on the public line information from `Line`.

## Extract all NeTEx Lines

The line sample showed that the NeTEx line files contain usable public line information.

I now extract all `Line` elements from the line-specific XML files. For each line, I keep the line ID, name, short name, public code, transport mode, and the number of related `Route` and `ServiceJourney` elements in the file.


In [26]:
def extract_all_lines(zip_path, file_list):
    rows = []
    total = len(file_list)

    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{total}] {member_name}", end="\r")
            file_lines = []
            route_count = 0
            service_journey_count = 0
            current_line = None
            line_depth = 0

            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "Line":
                        line_depth += 1
                        if line_depth == 1:
                            current_line = {
                                "file":                 member_name,
                                "netex_line_id":        elem.attrib.get("id"),
                                "netex_line_name":      None,
                                "netex_short_name":     None,
                                "netex_public_code":    None,
                                "netex_transport_mode": None
                            }

                    elif event == "end":
                        if tag == "Route":
                            route_count += 1
                        elif tag == "ServiceJourney":
                            service_journey_count += 1

                        if tag != "Line" and current_line is not None and line_depth > 0:
                            text = elem.text.strip() if elem.text else None
                            if tag == "Name" and current_line["netex_line_name"] is None:
                                current_line["netex_line_name"] = text
                            elif tag == "ShortName" and current_line["netex_short_name"] is None:
                                current_line["netex_short_name"] = text
                            elif tag == "PublicCode" and current_line["netex_public_code"] is None:
                                current_line["netex_public_code"] = text
                            elif tag == "TransportMode" and current_line["netex_transport_mode"] is None:
                                current_line["netex_transport_mode"] = text

                        if tag == "Line":
                            if line_depth == 1 and current_line is not None:
                                file_lines.append(current_line)
                                current_line = None
                            line_depth -= 1

                        elem.clear()

            for line in file_lines:
                line["route_count"] = route_count
                line["service_journey_count"] = service_journey_count
                rows.append(line)

    print(f"\nDone — {len(rows)} lines extracted from {total} files.")
    return pd.DataFrame(rows)

netex_lines = extract_all_lines(netex_zip_path, line_files)
display(netex_lines.head(20))

  [4421/4421] RUT_RUT-Line-3130_130N_Nationaltheatret---Sandvika.xmlgda---Oppdal.xmls---Alesund.xmlepiren.xmlxmlso.xmlbo.xml
Done — 4421 lines extracted from 4421 files.


,file,netex_line_id,netex_line_name,netex_short_name,netex_public_code,netex_transport_mode,route_count,service_journey_count
0,VKT_VKT-Line-1213_1213_Bugarden-Sagmyra.xml,VKT:Line:1213,Bugården-Sagmyra,None,1213,bus,2,4
1,INN_INN-Line-312_312_Lidar---Oystre-Slidre.xml,INN:Line:312,Lidar - Øystre Slidre,None,312,bus,2,22
2,OST_OST-Line-1_472_472_Skoler-Rakkestad.xml,OST:Line:1_472,Skoler Rakkestad,None,472,bus,4,4
3,KOL_KOL-Line-8_2643_SK2201_SK2201.xml,KOL:Line:8_2643,SK2201,None,SK2201,bus,3,4
4,SKY_SKY-Line-460E_460E_Agotnes---Bergen.xml,SKY:Line:460E,Ågotnes - Bergen,None,460E,bus,2,105
5,RUT_RUT-Line-5069_506X_Sommerbuss-i-Drobak.xml,RUT:Line:5069,Sommerbuss i Drøbak,None,506X,bus,2,44
6,OST_OST-Line-1_197_197_Hvaler-barne-og-ungdoms...,OST:Line:1_197,Hvaler-barne og ungdomsskole,None,197,bus,12,24
7,SOF_SOF-Line-6128_69_257_Sandvik-Eikefjord.xml,SOF:Line:6128_69,Sandvik-Eikefjord,None,257,bus,6,7
8,SKY_SKY-Line-920_920_Dale---Vaksdal---Arna---B...,SKY:Line:920,Dale - Vaksdal - Arna - Bergen,None,920,bus,8,14
9,KOL_KOL-Line-8_1018_34_34.xml,KOL:Line:8_1018,34,None,34,bus,15,58


In [27]:
# Check quality of extracted NeTEx lines

netex_line_quality = pd.DataFrame({
    "metric": [
        "Line rows",
        "unique Line IDs",
        "missing line names",
        "missing short names",
        "missing public codes",
        "missing transport modes",
        "duplicate Line IDs",
        "unique public codes"
    ],
    "value": [
        len(netex_lines),
        netex_lines["netex_line_id"].nunique(),
        netex_lines["netex_line_name"].isna().sum(),
        netex_lines["netex_short_name"].isna().sum(),
        netex_lines["netex_public_code"].isna().sum(),
        netex_lines["netex_transport_mode"].isna().sum(),
        netex_lines["netex_line_id"].duplicated().sum(),
        netex_lines["netex_public_code"].nunique()
    ]
})

display(netex_line_quality)

print("Transport mode distribution:")
display(
    netex_lines["netex_transport_mode"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "netex_transport_mode", "netex_transport_mode": "number_of_lines"})
)

,metric,value
0,Line rows,4421
1,unique Line IDs,4421
2,missing line names,0
3,missing short names,4386
4,missing public codes,119
5,missing transport modes,0
6,duplicate Line IDs,0
7,unique public codes,2461


Transport mode distribution:


,number_of_lines,count
0,bus,3916
1,water,286
2,air,106
3,rail,53
4,coach,44
5,tram,9
6,metro,5
7,cableway,2


## Comment

There are 4,421 NeTEx line rows, and each line has a unique `Line` ID. All lines have a line name and transport mode, and there are no duplicate line IDs.

The `ShortName` field is mostly missing, so it is not useful for the later comparison. The better NeTEx field for the public line label is `public_code`. However, 119 lines are missing a public code, so these will need to be treated like the GTFS routes without `route_short_name`.

The transport mode distribution shows that the NeTEx dataset is mainly bus, but it also includes water, air, rail, coach, tram, metro, and cableway services. This matches the multimodal structure already seen in the GTFS data.

## Inspect NeTEx lines without public codes

Some NeTEx lines do not have a `public_code`.  
Before preparing the final NeTEx line-level table, I inspect these rows separately to understand which transport modes they belong to.


In [28]:
# Inspect NeTEx lines without public_code

netex_lines_without_public_code = netex_lines[
    netex_lines["netex_public_code"].isna()
].copy()

missing_public_code_summary = (
    netex_lines_without_public_code
    .groupby("netex_transport_mode")
    .size()
    .reset_index(name="number_of_lines")
    .sort_values("number_of_lines", ascending=False)
)

display(missing_public_code_summary)

display(
    netex_lines_without_public_code[
        [
            "file",
            "netex_line_id",
            "netex_line_name",
            "netex_public_code",
            "netex_transport_mode",
            "route_count",
            "service_journey_count"
        ]
    ].head(20)
)

,netex_transport_mode,number_of_lines
0,air,106
1,bus,7
2,rail,4
3,water,2


,file,netex_line_id,netex_line_name,netex_public_code,netex_transport_mode,route_count,service_journey_count
55,AVI_AVI-Line-WF_TRD-MJF_Trondheim-Mosjoen.xml,AVI:Line:WF_TRD-MJF,Trondheim-Mosjøen,None,air,2,3
67,AVI_AVI-Line-WF_BGO-KSU_Bergen-Kristiansund.xml,AVI:Line:WF_BGO-KSU,Bergen-Kristiansund,None,air,2,15
203,AVI_AVI-Line-WF_TRD-RVK_Trondheim-Rorvik.xml,AVI:Line:WF_TRD-RVK,Trondheim-Rørvik,None,air,4,13
258,AVI_AVI-Line-WF_OSL-BGO_Oslo-Bergen.xml,AVI:Line:WF_OSL-BGO,Oslo-Bergen,None,air,6,15
273,AVI_AVI-Line-DY_OSL-EVE_Oslo-Harstad-Narvik.xml,AVI:Line:DY_OSL-EVE,Oslo-Harstad/Narvik,None,air,2,28
291,AVI_AVI-Line-DX_EVE-OLA_Harstad-Narvik-Orland.xml,AVI:Line:DX_EVE-OLA,Harstad/Narvik-Ørland,None,air,2,3
371,AVI_AVI-Line-WF_SKN-TOS_Stokmarknes-Tromso.xml,AVI:Line:WF_SKN-TOS,Stokmarknes-Tromsø,None,air,2,4
383,SAM_SAM-Line-9011012680800000__Ers-buss-Gotebo...,SAM:Line:9011012680800000,Ers.buss Göteborg - Halmstad,None,bus,16,131
386,AVI_AVI-Line-SK_BGO-TRD_Bergen-Trondheim.xml,AVI:Line:SK_BGO-TRD,Bergen-Trondheim,None,air,2,18
402,AVI_AVI-Line-WF_TOS-SOJ_Tromso-Sorkjosen.xml,AVI:Line:WF_TOS-SOJ,Tromsø-Sørkjosen,None,air,2,6


## Comment

Most missing public codes belong to air services: 106 out of the 119 missing-code lines are air lines. Only a small number belong to bus, rail, or water services.

This is very similar to what was seen on the GTFS side, where many missing `route_short_name` values were also air services. 

## Prepare NeTEx line-level dataset

The NeTEx line table contains technical `Line` rows, but the later line-level comparison should use the visible public line label.

For Norway, `ShortName` is mostly missing, so `netex_public_code` is the best NeTEx field for this purpose.  
Lines without `public_code` were already inspected separately, so I now prepare a deduplicated line-level table using only lines with a public code.

In [29]:
# Prepare NeTEx line-level dataset
netex_lines_with_code = netex_lines[netex_lines["netex_public_code"].notna()].copy()

netex_line_summary = pd.DataFrame({
    "metric": [
        "Line rows",
        "Line rows with public_code",
        "Line rows without public_code",
        "unique public_code values",
        "duplicate public_code rows"
    ],
    "value": [
        len(netex_lines),
        len(netex_lines_with_code),
        netex_lines["netex_public_code"].isna().sum(),
        netex_lines_with_code["netex_public_code"].nunique(),
        netex_lines_with_code["netex_public_code"].duplicated().sum()
    ]
})
display(netex_line_summary)

netex_lines_dedup = (
    netex_lines_with_code
    .groupby("netex_public_code", as_index=False)
    .agg(
        number_of_line_rows=("netex_line_id", "count"),
        transport_modes=("netex_transport_mode", lambda x: ", ".join(sorted(x.dropna().unique().tolist()))),
        example_line_name=("netex_line_name", "first"),
        # route_count and service_journey_count are file-level totals, so this sum
        # reflects the combined count across all files sharing this public_code
        route_count_total=("route_count", "sum"),
        service_journey_count_total=("service_journey_count", "sum")
    )
    .rename(columns={"netex_public_code": "netex_line_label"})
    .sort_values(
        "netex_line_label",
        key=lambda s: pd.to_numeric(s, errors="coerce").fillna(float("inf"))
    )
    .reset_index(drop=True)
)
display(netex_lines_dedup.head(20))

,metric,value
0,Line rows,4421
1,Line rows with public_code,4302
2,Line rows without public_code,119
3,unique public_code values,2461
4,duplicate public_code rows,1841


,netex_line_label,number_of_line_rows,transport_modes,example_line_name,route_count_total,service_journey_count_total
0,1,11,"bus, metro, tram",Håkvik - Sentrum - Universitetssykehuset,98,10327
1,001,1,water,Hyke Arendalsuka,2,13
2,01,1,bus,Horten-Tønsberg-Sandefjord,7,1200
3,2,12,"bus, metro, tram, water",Notodden-Sjukehuset-Anundskås,139,10295
4,02,4,bus,Tjøme-Tønsberg-Horten-Holmestr,13,2098
5,3,12,"bus, metro, water",Ålesund-Valderøya-Vigra,129,10258
6,4,10,"bus, metro, water",Notodden-Tuven-Stavkrk-Kvantum,85,8415
7,5,7,"bus, metro, water",Vinnes - Tors vei,67,6407
8,6,5,"bus, water",Ekspress Larvik-Tønsberg,46,3971
9,7,2,bus,Moum-Ambjørnrød,14,663


## Comment

The NeTEx line-level table is now prepared at the public-code level.

The original NeTEx line table contains 4,421 line rows. Out of these, 4,302 lines have a `public_code`, while 119 lines do not. After grouping by `public_code`, there are 2,461 unique public line labels.

There are 1,841 duplicate public-code rows, which means that many visible line labels appear in more than one technical NeTEx line row. This is similar to the GTFS side, where several `route_id` rows also shared the same `route_short_name`.

The table also shows that some public codes are connected to several transport modes, for example bus, metro, tram, or water. This means that the public line label is useful for comparison, but transport mode should be kept as context when interpreting the results later.

## Inspect NeTEx calendar structure

The goal is to check where `DayType`, `DayTypeAssignment`, and `OperatingPeriod` elements are stored. 

In [30]:
# Select NeTEx shared data files

shared_files = [f for f in netex_files["name"].tolist() if "shared_data" in f]

print("Number of shared data files:", len(shared_files))
print("Example shared data files:")
display(pd.DataFrame({"file": shared_files[:20]}))

Number of shared data files: 67
Example shared data files:


,file
0,_BOR_flexible_shared_data.xml
1,_AKT_shared_data.xml
2,_VYX_flexible_shared_data.xml
3,_MOR_shared_data.xml
4,_TEL_flexible_shared_data.xml
5,_BRA_shared_data.xml
6,_VYS_shared_data.xml
7,_TTS_shared_data.xml
8,_VKT_shared_data.xml
9,_TID_shared_data.xml


In [31]:
# Inspect calendar-related elements in shared data files

calendar_target_tags = {
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "UicOperatingPeriod",
    "FromDate",
    "ToDate",
    "ValidDayBits"
}

calendar_structure_rows = []

for member_name in shared_files:
    counts = scan_xml_elements_in_zip(
        netex_zip_path,
        member_name,
        calendar_target_tags
    )
    
    row = {"file": member_name}
    for tag in sorted(calendar_target_tags):
        row[tag] = counts.get(tag, 0)
    
    calendar_structure_rows.append(row)

netex_calendar_structure = pd.DataFrame(calendar_structure_rows)

display(
    netex_calendar_structure
    .sort_values(["DayTypeAssignment", "DayType", "OperatingPeriod"], ascending=False)
)

,file,DayType,DayTypeAssignment,FromDate,OperatingPeriod,ToDate,UicOperatingPeriod,ValidDayBits
42,_SKY_shared_data.xml,1159,13634,1,0,1,0,0
55,_AVI_shared_data.xml,7882,7882,1,0,1,0,0
65,_KOL_shared_data.xml,433,7468,326,325,326,0,0
22,_VYX_shared_data.xml,910,6711,911,910,911,0,0
62,_NOR_shared_data.xml,296,6431,272,271,272,0,0
...,...,...,...,...,...,...,...,...
48,_RUT_shared_data.xml,0,0,1,0,1,0,0
53,_BAR_flexible_shared_data.xml,0,0,1,0,1,0,0
59,_SJN_shared_data.xml,0,0,1,0,1,0,0
60,_SAM_shared_data.xml,0,0,1,0,1,0,0


## Comment

The Norway NeTEx dataset contains **67 shared data files**, one for each public transport operator.
Each file is named after a three-letter operator code, for example `_RUT_shared_data.xml` for
Ruter (Oslo) or `_SKY_shared_data.xml` for Skyss (Bergen).

Some files have `flexible` in their name (e.g. `_BOR_flexible_shared_data.xml`). These cover
**demand-responsive or flexible transport** services (routes that do not run on a fixed schedule
but can be booked on request). The remaining files cover standard fixed-schedule services.

This operator-by-operator structure is different from the other countries analysed so far.
In Luxembourg and France, the NeTEx data came as a single national file or a small set of files.
In Norway, each operator manages its own file, which means the data is more fragmented but also
more clearly separated by authority.

## Inspect NeTEx calendar element samples

The structure check showed that Norway does not seem to use `UicOperatingPeriod` or `ValidDayBits`.  
Instead, the calendar information appears to use `DayTypeAssignment`, `OperatingPeriod`, `FromDate`, and `ToDate`.

Before extracting the full NeTEx calendar data, I inspect a small sample of these elements to understand how the validity information is represented.

In [32]:
# Select shared files that contain calendar assignments

calendar_shared_files = (
    netex_calendar_structure[netex_calendar_structure["DayTypeAssignment"] > 0]
    .sort_values("DayTypeAssignment", ascending=False)["file"]
    .tolist()
)

print("Number of shared files with DayTypeAssignment:", len(calendar_shared_files))
print("Example files:")
display(pd.DataFrame({"file": calendar_shared_files[:10]}))

Number of shared files with DayTypeAssignment: 56
Example files:


,file
0,_SKY_shared_data.xml
1,_AVI_shared_data.xml
2,_KOL_shared_data.xml
3,_VYX_shared_data.xml
4,_NOR_shared_data.xml
5,_ATB_flexible_shared_data.xml
6,_SOF_shared_data.xml
7,_FIN_shared_data.xml
8,_BRA_shared_data.xml
9,_TRO_shared_data.xml


In [33]:
def extract_calendar_sample(zip_path, file_list, max_files=3, max_assignments=20, max_periods=20):
    assignment_rows = []
    period_rows = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list[:max_files], start=1):
            print(f"  [{i}/{min(max_files, len(file_list))}] {member_name}", end="\r")

            current_assignment = None
            assignment_depth = 0
            current_period = None
            period_depth = 0

            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    # --- DayTypeAssignment ---
                    if event == "start" and tag == "DayTypeAssignment":
                        assignment_depth += 1
                        if assignment_depth == 1:
                            current_assignment = {
                                "file":                  member_name,
                                "assignment_id":         elem.attrib.get("id"),
                                "daytype_ref":           None,
                                "operating_period_ref":  None,
                                "date":                  None,
                                "is_available":          None
                            }

                    elif event == "end" and tag == "DayTypeAssignment":
                        if assignment_depth == 1 and current_assignment is not None:
                            assignment_rows.append(current_assignment)
                            current_assignment = None
                        assignment_depth -= 1
                        elem.clear()

                    elif event == "end" and current_assignment is not None and assignment_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        # DayTypeRef and OperatingPeriodRef are self-closing elements
                        # (<DayTypeRef ref="..."/>) so the value is in the attribute, not the text
                        if tag == "DayTypeRef" and current_assignment["daytype_ref"] is None:
                            current_assignment["daytype_ref"] = elem.attrib.get("ref")
                        elif tag == "OperatingPeriodRef" and current_assignment["operating_period_ref"] is None:
                            current_assignment["operating_period_ref"] = elem.attrib.get("ref")
                        elif tag == "Date" and current_assignment["date"] is None:
                            current_assignment["date"] = text
                        elif tag == "IsAvailable" and current_assignment["is_available"] is None:
                            current_assignment["is_available"] = text
                        elem.clear()

                    # --- OperatingPeriod ---
                    elif event == "start" and tag == "OperatingPeriod":
                        period_depth += 1
                        if period_depth == 1:
                            current_period = {
                                "file":                member_name,
                                "operating_period_id": elem.attrib.get("id"),
                                "from_date":           None,
                                "to_date":             None
                            }

                    elif event == "end" and tag == "OperatingPeriod":
                        if period_depth == 1 and current_period is not None:
                            period_rows.append(current_period)
                            current_period = None
                        period_depth -= 1
                        elem.clear()

                    elif event == "end" and current_period is not None and period_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "FromDate" and current_period["from_date"] is None:
                            current_period["from_date"] = text
                        elif tag == "ToDate" and current_period["to_date"] is None:
                            current_period["to_date"] = text
                        elem.clear()

                    elif event == "end":
                        elem.clear()

            if len(assignment_rows) >= max_assignments and len(period_rows) >= max_periods:
                break

    print(f"\nDone — {len(assignment_rows)} assignments, {len(period_rows)} periods collected.")
    return (
        pd.DataFrame(assignment_rows).head(max_assignments),
        pd.DataFrame(period_rows).head(max_periods)
    )

netex_daytype_assignment_sample, netex_operating_period_sample = extract_calendar_sample(
    netex_zip_path,
    calendar_shared_files,
    max_files=3,
    max_assignments=20,
    max_periods=20
)

print("DayTypeAssignment sample:")
display(netex_daytype_assignment_sample)

print("OperatingPeriod sample:")
display(netex_operating_period_sample)

  [3/3] _KOL_shared_data.xml
Done — 28984 assignments, 325 periods collected.
DayTypeAssignment sample:


,file,assignment_id,daytype_ref,operating_period_ref,date,is_available
0,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-1,SKY:DayType:167695-0111110,None,2026-05-08,None
1,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-10,SKY:DayType:167695-0111110,None,2026-05-22,None
2,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-11,SKY:DayType:167695-0111110,None,2026-05-26,None
3,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-12,SKY:DayType:167695-0111110,None,2026-05-27,None
4,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-13,SKY:DayType:167695-0111110,None,2026-05-28,None
5,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-14,SKY:DayType:167695-0111110,None,2026-05-29,None
6,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-15,SKY:DayType:167695-0111110,None,2026-06-01,None
7,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-16,SKY:DayType:167695-0111110,None,2026-06-02,None
8,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-17,SKY:DayType:167695-0111110,None,2026-06-03,None
9,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-18,SKY:DayType:167695-0111110,None,2026-06-04,None


OperatingPeriod sample:


,file,operating_period_id,from_date,to_date
0,_KOL_shared_data.xml,KOL:OperatingPeriod:1328-1,2026-04-30T00:00:00,2027-01-05T00:00:00
1,_KOL_shared_data.xml,KOL:OperatingPeriod:1283-1,2026-08-15T00:00:00,2026-12-22T00:00:00
2,_KOL_shared_data.xml,KOL:OperatingPeriod:1512-1,2026-08-12T00:00:00,2026-12-21T00:00:00
3,_KOL_shared_data.xml,KOL:OperatingPeriod:1535-1,2026-05-15T00:00:00,2026-12-31T00:00:00
4,_KOL_shared_data.xml,KOL:OperatingPeriod:1389-1,2026-05-05T00:00:00,2026-12-22T00:00:00
5,_KOL_shared_data.xml,KOL:OperatingPeriod:1366-1,2026-05-05T00:00:00,2026-12-23T00:00:00
6,_KOL_shared_data.xml,KOL:OperatingPeriod:1305-1,2026-05-02T00:00:00,2026-12-24T00:00:00
7,_KOL_shared_data.xml,KOL:OperatingPeriod:1558-1,2026-05-02T00:00:00,2026-12-24T00:00:00
8,_KOL_shared_data.xml,KOL:OperatingPeriod:1222-1,2026-07-01T00:00:00,2026-12-30T00:00:00
9,_KOL_shared_data.xml,KOL:OperatingPeriod:1475-1,2026-04-29T00:00:00,2026-12-21T00:00:00


## Comment

The NeTEx calendar sample shows that Norway mainly uses direct date assignments.

There are 56 shared files with `DayTypeAssignment` elements. In the sample, the `DayTypeAssignment` rows contain a `daytype_ref` and a specific `date`, but no `operating_period_ref`. This means many NeTEx service patterns are represented by explicit operating dates.

The `OperatingPeriod` elements also exist, but in this sample they appear separately, with `from_date` and `to_date`. They are not directly referenced in the shown `DayTypeAssignment` rows.

This is important because Norway does not seem to use the `ValidDayBits` approach. For Norway, the NeTEx calendar comparison will probably need to be based mainly on the explicit `DayTypeAssignment` dates.

## Extract NeTEx DayTypeAssignments

The sample showed that many NeTEx calendar assignments contain a direct `date` linked to a `DayType`.

I now extract all `DayTypeAssignment` elements from the shared data files.  
This prepares the NeTEx calendar dataset and helps check how many date-based assignments exist.

In [34]:
# Extract all DayTypeAssignment elements from shared data files
def extract_all_daytype_assignments(zip_path, file_list):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{len(file_list)}] {member_name}", end="\r")
            current_assignment = None
            assignment_depth = 0
            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "DayTypeAssignment":
                        assignment_depth += 1
                        if assignment_depth == 1:
                            current_assignment = {
                                "file":                 member_name,
                                "assignment_id":        elem.attrib.get("id"),
                                "daytype_ref":          None,
                                "operating_period_ref": None,
                                "date":                 None,
                                "is_available":         None
                            }

                    elif event == "end" and tag == "DayTypeAssignment":
                        if assignment_depth == 1 and current_assignment is not None:
                            rows.append(current_assignment)
                            current_assignment = None
                        assignment_depth -= 1
                        elem.clear()

                    elif event == "end" and current_assignment is not None and assignment_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        # DayTypeRef and OperatingPeriodRef are self-closing elements
                        # (<DayTypeRef ref="..."/>) so the value is in the attribute, not the text
                        if tag == "DayTypeRef" and current_assignment["daytype_ref"] is None:
                            current_assignment["daytype_ref"] = elem.attrib.get("ref")
                        elif tag == "OperatingPeriodRef" and current_assignment["operating_period_ref"] is None:
                            current_assignment["operating_period_ref"] = elem.attrib.get("ref")
                        elif tag == "Date" and current_assignment["date"] is None:
                            current_assignment["date"] = text
                        elif tag == "IsAvailable" and current_assignment["is_available"] is None:
                            current_assignment["is_available"] = text
                        elem.clear()

                    elif event == "end":
                        elem.clear()

    print(f"\nDone — {len(rows)} DayTypeAssignments extracted.")
    return pd.DataFrame(rows)

netex_daytype_assignments = extract_all_daytype_assignments(
    netex_zip_path,
    calendar_shared_files
)

# Convert date strings to datetime and is_available to boolean
netex_daytype_assignments["date"] = pd.to_datetime(
    netex_daytype_assignments["date"],
    errors="coerce"
)
netex_daytype_assignments["is_available"] = netex_daytype_assignments["is_available"].map(
    {"true": True, "false": False}
)

display(netex_daytype_assignments.head(20))

  [56/56] _VYX_flexible_shared_data.xml
Done — 72612 DayTypeAssignments extracted.


,file,assignment_id,daytype_ref,operating_period_ref,date,is_available
0,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-1,SKY:DayType:167695-0111110,None,2026-05-08,NaN
1,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-10,SKY:DayType:167695-0111110,None,2026-05-22,NaN
2,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-11,SKY:DayType:167695-0111110,None,2026-05-26,NaN
3,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-12,SKY:DayType:167695-0111110,None,2026-05-27,NaN
4,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-13,SKY:DayType:167695-0111110,None,2026-05-28,NaN
5,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-14,SKY:DayType:167695-0111110,None,2026-05-29,NaN
6,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-15,SKY:DayType:167695-0111110,None,2026-06-01,NaN
7,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-16,SKY:DayType:167695-0111110,None,2026-06-02,NaN
8,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-17,SKY:DayType:167695-0111110,None,2026-06-03,NaN
9,_SKY_shared_data.xml,SKY:DayTypeAssignment:167695-0111110-18,SKY:DayType:167695-0111110,None,2026-06-04,NaN


In [35]:
netex_daytype_assignment_quality = pd.DataFrame({
    "metric": [
        "DayTypeAssignment rows",
        "unique assignment IDs",
        "unique daytype_ref values",
        "missing daytype_ref",
        "missing dates",
        "rows with operating_period_ref",
        "earliest date",
        "latest date"
    ],
    "value": [
        len(netex_daytype_assignments),
        netex_daytype_assignments["assignment_id"].nunique(),
        netex_daytype_assignments["daytype_ref"].nunique(),
        netex_daytype_assignments["daytype_ref"].isna().sum(),
        netex_daytype_assignments["date"].isna().sum(),
        netex_daytype_assignments["operating_period_ref"].notna().sum(),
        netex_daytype_assignments["date"].min(),
        netex_daytype_assignments["date"].max()
    ]
})

display(netex_daytype_assignment_quality)

print("is_available distribution:")
display(
    netex_daytype_assignments["is_available"]
    .fillna("missing")
    .value_counts()
    .reset_index()
    .rename(columns={"index": "is_available", "is_available": "count"})
)

,metric,value
0,DayTypeAssignment rows,72612
1,unique assignment IDs,72612
2,unique daytype_ref values,15773
3,missing daytype_ref,0
4,missing dates,13424
5,rows with operating_period_ref,13424
6,earliest date,2019-04-19 00:00:00
7,latest date,2027-03-30 00:00:00


is_available distribution:


,count,count
0,missing,72612


## Comment

The NeTEx DayTypeAssignment extraction worked successfully.

In total, 72,612 DayTypeAssignment rows were extracted. All rows have a unique assignment ID and a `daytype_ref`, so the link to DayType is available.

Most rows contain a direct `date`, which means they can be used as explicit active dates. However, 13,424 rows do not have a direct date. These same rows have an `operating_period_ref`, so their validity is probably defined through an OperatingPeriod instead of a single date.

The `is_available` field is missing for all rows, so it cannot be used to distinguish added or removed days. For the next step, the rows with `operating_period_ref` need to be checked together with the OperatingPeriod table.

## Extract NeTEx OperatingPeriods

Some `DayTypeAssignment` rows do not contain a direct date.  
Instead, they refer to an `OperatingPeriod`.

I now extract all `OperatingPeriod` elements from the shared NeTEx files.  
This helps complete the NeTEx calendar information before building the final calendar patterns.

In [36]:
# Extract all OperatingPeriod elements from shared data files
def extract_all_operating_periods(zip_path, file_list):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{len(file_list)}] {member_name}", end="\r")
            current_period = None
            period_depth = 0
            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "OperatingPeriod":
                        period_depth += 1
                        if period_depth == 1:
                            current_period = {
                                "file":                member_name,
                                "operating_period_id": elem.attrib.get("id"),
                                "from_date":           None,
                                "to_date":             None
                            }
                    elif event == "end" and tag == "OperatingPeriod":
                        if period_depth == 1 and current_period is not None:
                            rows.append(current_period)
                            current_period = None
                        period_depth -= 1
                        elem.clear()
                    elif event == "end" and current_period is not None and period_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "FromDate" and current_period["from_date"] is None:
                            current_period["from_date"] = text
                        elif tag == "ToDate" and current_period["to_date"] is None:
                            current_period["to_date"] = text
                        elem.clear()
                    elif event == "end":
                        elem.clear()

    print(f"\nDone — {len(rows)} OperatingPeriods extracted.")
    return pd.DataFrame(rows)

netex_operating_periods = extract_all_operating_periods(
    netex_zip_path,
    calendar_shared_files
)

netex_operating_periods["from_date"] = pd.to_datetime(
    netex_operating_periods["from_date"],
    errors="coerce"
)
netex_operating_periods["to_date"] = pd.to_datetime(
    netex_operating_periods["to_date"],
    errors="coerce"
)

display(netex_operating_periods.head(20))

  [56/56] _VYX_flexible_shared_data.xml
Done — 13274 OperatingPeriods extracted.


,file,operating_period_id,from_date,to_date
0,_KOL_shared_data.xml,KOL:OperatingPeriod:1328-1,2026-04-30,2027-01-05
1,_KOL_shared_data.xml,KOL:OperatingPeriod:1283-1,2026-08-15,2026-12-22
2,_KOL_shared_data.xml,KOL:OperatingPeriod:1512-1,2026-08-12,2026-12-21
3,_KOL_shared_data.xml,KOL:OperatingPeriod:1535-1,2026-05-15,2026-12-31
4,_KOL_shared_data.xml,KOL:OperatingPeriod:1389-1,2026-05-05,2026-12-22
5,_KOL_shared_data.xml,KOL:OperatingPeriod:1366-1,2026-05-05,2026-12-23
6,_KOL_shared_data.xml,KOL:OperatingPeriod:1305-1,2026-05-02,2026-12-24
7,_KOL_shared_data.xml,KOL:OperatingPeriod:1558-1,2026-05-02,2026-12-24
8,_KOL_shared_data.xml,KOL:OperatingPeriod:1222-1,2026-07-01,2026-12-30
9,_KOL_shared_data.xml,KOL:OperatingPeriod:1475-1,2026-04-29,2026-12-21


In [37]:
# Check OperatingPeriod quality

netex_operating_period_quality = pd.DataFrame({
    "metric": [
        "OperatingPeriod rows",
        "unique OperatingPeriod IDs",
        "missing OperatingPeriod IDs",
        "missing from_date",
        "missing to_date",
        "duplicate OperatingPeriod IDs",
        "earliest from_date",
        "latest to_date"
    ],
    "value": [
        len(netex_operating_periods),
        netex_operating_periods["operating_period_id"].nunique(),
        netex_operating_periods["operating_period_id"].isna().sum(),
        netex_operating_periods["from_date"].isna().sum(),
        netex_operating_periods["to_date"].isna().sum(),
        netex_operating_periods["operating_period_id"].duplicated().sum(),
        netex_operating_periods["from_date"].min(),
        netex_operating_periods["to_date"].max()
    ]
})

display(netex_operating_period_quality)

# Check whether DayTypeAssignment operating_period_ref values are found
assignment_period_refs = set(
    netex_daytype_assignments["operating_period_ref"].dropna()
)

period_ids = set(
    netex_operating_periods["operating_period_id"].dropna()
)

period_reference_check = pd.DataFrame({
    "metric": [
        "unique operating_period_ref values in DayTypeAssignment",
        "operating_period_ref values found in OperatingPeriod table",
        "operating_period_ref values missing from OperatingPeriod table"
    ],
    "value": [
        len(assignment_period_refs),
        len(assignment_period_refs & period_ids),
        len(assignment_period_refs - period_ids)
    ]
})

display(period_reference_check)

,metric,value
0,OperatingPeriod rows,13274
1,unique OperatingPeriod IDs,13274
2,missing OperatingPeriod IDs,0
3,missing from_date,366
4,missing to_date,366
5,duplicate OperatingPeriod IDs,0
6,earliest from_date,2019-04-13 00:00:00
7,latest to_date,2027-12-31 00:00:00


,metric,value
0,unique operating_period_ref values in DayTypeA...,13274
1,operating_period_ref values found in Operating...,13274
2,operating_period_ref values missing from Opera...,0


## Comment

The OperatingPeriod extraction worked well.

In total, 13,274 OperatingPeriod rows were extracted, and all OperatingPeriod IDs are unique. Most periods have both `from_date` and `to_date`, although 366 rows are missing these date values.

The reference check is very important: all 13,274 `operating_period_ref` values from `DayTypeAssignment` are found in the OperatingPeriod table. This means the DayTypeAssignment rows without a direct date can be linked correctly to their validity periods.

So the NeTEx calendar side contains two types of validity information: direct date assignments and assignments linked through OperatingPeriods.

## Build NeTEx active dates

The NeTEx calendar data uses two forms of validity information:

- DayTypeAssignment rows `with` a direct date
- DayTypeAssignment rows `without` date, but with operating_period_ref 

This step creates one combined NeTEx active-date table. Direct date assignments are used as they are, while OperatingPeriods are expanded into individual dates between `from_date` and `to_date`.

The result will later allow a fairer calendar comparison with GTFS, because both sides can be compared as active-day patterns.

In [38]:
# Build NeTEx active dates from direct dates and OperatingPeriods

# Only keep assignments where the day type is active
# is_available = False means the day is explicitly cancelled; null or True means active
available_assignments = netex_daytype_assignments[
    netex_daytype_assignments["is_available"].isna()
    | (netex_daytype_assignments["is_available"] == True)
].copy()

# 1. Direct date assignments
netex_direct_dates = (
    available_assignments[
        available_assignments["date"].notna()
    ][["daytype_ref", "date"]]
    .copy()
    .rename(columns={"date": "active_date"})
)
netex_direct_dates["calendar_source"] = "direct date"

# 2. Assignments linked to OperatingPeriod
netex_period_assignments = available_assignments[
    available_assignments["date"].isna()
    & available_assignments["operating_period_ref"].notna()
].copy()

netex_period_assignments = netex_period_assignments.merge(
    netex_operating_periods[["operating_period_id", "from_date", "to_date"]],
    left_on="operating_period_ref",
    right_on="operating_period_id",
    how="left"
)

# Keep only rows where the period has usable dates
netex_period_assignments_valid = netex_period_assignments[
    netex_period_assignments["from_date"].notna()
    & netex_period_assignments["to_date"].notna()
].copy()

# Expand OperatingPeriods into individual active dates
period_date_rows = []
for _, row in netex_period_assignments_valid.iterrows():
    for d in pd.date_range(row["from_date"], row["to_date"], freq="D"):
        period_date_rows.append({
            "daytype_ref":     row["daytype_ref"],
            "active_date":     d,
            "calendar_source": "operating period"
        })
netex_period_dates = pd.DataFrame(period_date_rows)

# 3. Combine both sources and deduplicate on daytype + date
netex_active_dates = pd.concat(
    [netex_direct_dates, netex_period_dates],
    ignore_index=True
).drop_duplicates(subset=["daytype_ref", "active_date"])

display(netex_active_dates.head(20))

netex_active_date_summary = pd.DataFrame({
    "metric": [
        "active-date rows",
        "unique daytype_ref values",
        "earliest active date",
        "latest active date",
        "direct date rows",
        "operating period date rows"
    ],
    "value": [
        len(netex_active_dates),
        netex_active_dates["daytype_ref"].nunique(),
        netex_active_dates["active_date"].min(),
        netex_active_dates["active_date"].max(),
        len(netex_direct_dates),
        len(netex_period_dates)
    ]
})
display(netex_active_date_summary)

,daytype_ref,active_date,calendar_source
0,SKY:DayType:167695-0111110,2026-05-08,direct date
1,SKY:DayType:167695-0111110,2026-05-22,direct date
2,SKY:DayType:167695-0111110,2026-05-26,direct date
3,SKY:DayType:167695-0111110,2026-05-27,direct date
4,SKY:DayType:167695-0111110,2026-05-28,direct date
5,SKY:DayType:167695-0111110,2026-05-29,direct date
6,SKY:DayType:167695-0111110,2026-06-01,direct date
7,SKY:DayType:167695-0111110,2026-06-02,direct date
8,SKY:DayType:167695-0111110,2026-06-03,direct date
9,SKY:DayType:167695-0111110,2026-06-04,direct date


,metric,value
0,active-date rows,869931
1,unique daytype_ref values,15524
2,earliest active date,2019-04-13 00:00:00
3,latest active date,2027-12-31 00:00:00
4,direct date rows,59188
5,operating period date rows,1346304


## Comment

The table combines two sources: direct date assignments and dates expanded from OperatingPeriods. After deduplication, there are 869,931 active day rows for 15,524 unique `daytype_ref` values.

The active dates range from 2019 to 2027, which is similar to the broad date range already seen in GTFS. Most generated rows come from OperatingPeriods, because each OperatingPeriod is expanded into many individual dates.

This gives a practical calendar representation: one row per `daytype_ref` and active date. It can later be compared with the reconstructed GTFS active-day patterns in Part 3.
    
> **_NOTE:_**  operating period date rows number is larger than the final active-date rows because the final table removes duplicate daytype_ref + date combinations    

## Prepare NeTEx calendar summary

The NeTEx active-date table now contains one row per `daytype_ref` and active date.

I now summarize this table to one row per `daytype_ref`.  
For each NeTEx day type, I keep the number of active days, the first active date, the last active date, and the source of the calendar information.

In [39]:
# Prepare NeTEx calendar summary

netex_active_dates["active_date"] = pd.to_datetime(
    netex_active_dates["active_date"],
    errors="coerce"
).dt.normalize()

netex_calendar_summary = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .groupby("daytype_ref", as_index=False)
    .agg(
        n_active_days=("active_date", "nunique"),
        first_active_date=("active_date", "min"),
        last_active_date=("active_date", "max"),
        calendar_sources=("calendar_source", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .sort_values("daytype_ref")
    .reset_index(drop=True)
)

display(netex_calendar_summary.head(20))

netex_calendar_summary_quality = pd.DataFrame({
    "metric": [
        "calendar summary rows",
        "daytype_ref with only 1 active day",
        "missing first_active_date",
        "missing last_active_date",
        "earliest first_active_date",
        "latest last_active_date"
    ],
    "value": [
        len(netex_calendar_summary),
        netex_calendar_summary["n_active_days"].eq(1).sum(),
        netex_calendar_summary["first_active_date"].isna().sum(),
        netex_calendar_summary["last_active_date"].isna().sum(),
        netex_calendar_summary["first_active_date"].min(),
        netex_calendar_summary["last_active_date"].max()
    ]
})

display(netex_calendar_summary_quality)

,daytype_ref,n_active_days,first_active_date,last_active_date,calendar_sources
0,AKT:DayType:100_1,153,2026-05-06,2026-10-05,"direct date, operating period"
1,AKT:DayType:101_1,152,2026-05-02,2026-09-30,"direct date, operating period"
2,AKT:DayType:102_1,145,2026-05-06,2026-09-27,"direct date, operating period"
3,AKT:DayType:103_1,7,2026-05-15,2026-10-02,direct date
4,AKT:DayType:104_1,8,2026-05-15,2026-10-02,direct date
5,AKT:DayType:105_1,146,2026-05-05,2026-09-27,"direct date, operating period"
6,AKT:DayType:106_1,152,2026-05-06,2026-10-04,"direct date, operating period"
7,AKT:DayType:107_1,153,2026-05-09,2026-10-08,"direct date, operating period"
8,AKT:DayType:108_1,53,2026-08-13,2026-10-04,operating period
9,AKT:DayType:109_1,46,2026-08-13,2026-09-27,operating period


,metric,value
0,calendar summary rows,15524
1,daytype_ref with only 1 active day,8767
2,missing first_active_date,0
3,missing last_active_date,0
4,earliest first_active_date,2019-04-13 00:00:00
5,latest last_active_date,2027-12-31 00:00:00


## Comment

There are 15,524 unique `daytype_ref` values in the NeTEx calendar summary. Each row now represents one NeTEx day type with the number of active days, the first active date, the last active date, and the source of the calendar information.

The table shows that Norway uses a mix of direct date assignments and operating periods. Some day types are built from both sources, while others come only from direct dates or only from operating periods.

A large number of day types have only one active day: 8,767. This shows that many NeTEx calendar patterns are very specific date-based patterns.

There are no missing first or last active dates, so the NeTEx calendar summary is complete and ready for the later GTFS–NeTEx calendar comparison in Part 3.

# Part 3: GTFS–NeTEx Comparison

In this part, I compare the prepared GTFS and NeTEx datasets.

The comparison follows the same order as in the previous country notebooks:

1. station / stop level  
2. line / route level  
3. calendar / validity level  

I start with the station level because both GTFS and NeTEx use `NSR:StopPlace` identifiers.

## 3.1 Station-level comparison

The GTFS station-level table was built from `location_type = 1` rows.  
The NeTEx station-level table was built from `StopPlace` elements in `_stops.xml`.

Since both datasets use `NSR:StopPlace` IDs, I first compare stations by ID.

In [40]:
 # Station-level GTFS–NeTEx comparison by StopPlace ID
station_match = gtfs_stations.merge(
    netex_stopplaces,
    left_on="gtfs_stopplace_id",
    right_on="netex_stopplace_id",
    how="left",
    indicator=True
)

gtfs_matched  = (station_match["_merge"] == "both").sum()
netex_matched = station_match[station_match["_merge"] == "both"]["netex_stopplace_id"].nunique()

station_match_summary = pd.DataFrame({
    "metric": [
        "GTFS station rows",
        "NeTEx StopPlace rows",
        "matched GTFS stations",
        "unmatched GTFS stations",
        "GTFS-side match rate (%)",
        "NeTEx-side coverage rate (%)"
    ],
    "value": [
        len(gtfs_stations),
        len(netex_stopplaces),
        gtfs_matched,
        (station_match["_merge"] == "left_only").sum(),
        round(gtfs_matched  / len(gtfs_stations)    * 100, 2),
        round(netex_matched / len(netex_stopplaces) * 100, 2)
    ]
})
display(station_match_summary)

display(
    station_match[[
        "gtfs_stopplace_id",
        "gtfs_stop_name",
        "gtfs_lat",
        "gtfs_lon",
        "netex_stopplace_name",
        "netex_lat",
        "netex_lon",
        "_merge"
    ]].head(20)
)

,metric,value
0,GTFS station rows,51529.00
1,NeTEx StopPlace rows,57957.00
2,matched GTFS stations,51529.00
3,unmatched GTFS stations,0.00
4,GTFS-side match rate (%),100.00
5,NeTEx-side coverage rate (%),88.91


,gtfs_stopplace_id,gtfs_stop_name,gtfs_lat,gtfs_lon,netex_stopplace_name,netex_lat,netex_lon,_merge
0,NSR:StopPlace:19881,Sigurdsbu,59.677541,7.546677,Sigurdsbu,59.677541,7.546677,both
1,NSR:StopPlace:19948,Haukeli,59.735009,7.550864,Haukeli,59.735009,7.550864,both
2,NSR:StopPlace:21279,Sæsvatn,59.658592,7.502200,Sæsvatn,59.658592,7.502200,both
3,NSR:StopPlace:21367,Vamark,59.714885,7.577299,Vamark,59.714885,7.577299,both
4,NSR:StopPlace:21523,Stokketjønn,59.667051,7.529256,Stokketjønn,59.667051,7.529256,both
5,NSR:StopPlace:21717,Tvøråsen,58.327245,6.998855,Tvøråsen,58.327245,6.998855,both
6,NSR:StopPlace:21718,Ravnås,58.232717,7.931170,Ravnås,58.232717,7.931170,both
7,NSR:StopPlace:21719,Ravnåsvegen 347,58.252313,7.950407,Ravnåsvegen 347,58.252313,7.950407,both
8,NSR:StopPlace:21721,Hageveien,58.647459,9.107478,Hageveien,58.647459,9.107478,both
9,NSR:StopPlace:21722,Reddal,58.340942,8.458131,Reddal,58.340942,8.458131,both


## Comment

All 51,529 GTFS station rows matched a NeTEx `StopPlace` by their shared `NSR:StopPlace` ID,
giving a **100% GTFS-side match rate**.

The NeTEx-side coverage is lower at **88.91%** because NeTEx contains more StopPlaces than
GTFS exposes at station level (every GTFS station exists in NeTEx, but not the other way around).

Names and coordinates are almost identical for all matched stations.

In [41]:
# Coordinate check for matched stations
matched = station_match[station_match["_merge"] == "both"].copy()

matched["lat_diff"] = (matched["gtfs_lat"].astype(float) - matched["netex_lat"].astype(float)).abs()
matched["lon_diff"] = (matched["gtfs_lon"].astype(float) - matched["netex_lon"].astype(float)).abs()
matched["coord_exact"] = (matched["lat_diff"] == 0) & (matched["lon_diff"] == 0)

coord_summary = pd.DataFrame({
    "metric": [
        "matched stations checked",
        "exact coordinate match",
        "coordinate mismatch",
        "exact match rate (%)",
        "max latitude difference",
        "max longitude difference"
    ],
    "value": [
        len(matched),
        matched["coord_exact"].sum(),
        (~matched["coord_exact"]).sum(),
        round(matched["coord_exact"].mean() * 100, 2),
        matched["lat_diff"].max(),
        matched["lon_diff"].max()
    ]
})
display(coord_summary)

,metric,value
0,matched stations checked,51529.000000
1,exact coordinate match,51520.000000
2,coordinate mismatch,9.000000
3,exact match rate (%),99.980000
4,max latitude difference,0.001123
5,max longitude difference,0.000993


In [42]:
display(matched[~matched["coord_exact"]].sort_values("lat_diff", ascending=False).head(10))

,gtfs_stopplace_id,gtfs_stop_name,gtfs_lat,gtfs_lon,netex_stopplace_id,netex_stopplace_name,netex_lat,netex_lon,_merge,lat_diff,lon_diff,coord_exact
50094,NSR:StopPlace:18291,Holtandalen Kiwi,59.416809,10.460438,NSR:StopPlace:18291,Holtandalen Kiwi,59.415686,10.461431,both,1.123000e-03,0.000993,False
12217,NSR:StopPlace:28100,Preikestolhytta,58.991420,6.137380,NSR:StopPlace:28100,Preikestolhytta,58.991701,6.137860,both,2.810000e-04,0.000480,False
23908,NSR:StopPlace:40113,Sundeskiftet,62.417224,6.328075,NSR:StopPlace:40113,Sundeskiftet,62.417490,6.327910,both,2.660000e-04,0.000165,False
24950,NSR:StopPlace:41550,Solavågen ferjekai,62.413895,6.327527,NSR:StopPlace:41550,Solavågen ferjekai,62.413704,6.328230,both,1.910000e-04,0.000703,False
25166,NSR:StopPlace:62033,Solavågen ferjekai,62.414101,6.327828,NSR:StopPlace:62033,Solavågen ferjekai,62.414125,6.327929,both,2.400000e-05,0.000101,False
49314,NSR:StopPlace:58983,Ulriksbanen nedre stopp,60.373944,5.363747,NSR:StopPlace:58983,Ulriken stasjon Haukelandsbakken,60.373953,5.363774,both,9.000000e-06,0.000027,False
49313,NSR:StopPlace:61090,Scandic Harstad,68.801200,16.542322,NSR:StopPlace:61090,Scandic Harstad,68.801202,16.542323,both,2.000000e-06,0.000001,False
50098,NSR:StopPlace:18297,Holtandalen Ravnås,59.423767,10.454595,NSR:StopPlace:18297,Holtandalen Ravnås,59.423769,10.454618,both,2.000000e-06,0.000023,False
12216,NSR:StopPlace:27771,Tau kai,59.065320,5.907933,NSR:StopPlace:27771,Tau kai,59.065319,5.907933,both,1.000000e-06,0.000000,False


## Comment

The coordinate check confirms that the station-level match is very strong.

Out of 51,529 matched stations, 51,520 have exactly identical GTFS and NeTEx coordinates. Only 9 stations have small coordinate differences, giving an exact coordinate match rate of 99.98%.

GTFS and NeTEx use the same `NSR:StopPlace` identifiers, and almost all matched stations also share exactly the same coordinates.

## 3.2 Line-level comparison

After the station-level comparison, I now compare the GTFS and NeTEx line-level datasets.

On the GTFS side, the practical public line label is `route_short_name`.  
On the NeTEx side, the best available public line label is `public_code`.

Both sides were already deduplicated at the public-label level. I now compare these labels directly.

## Line-level comparison: GTFS vs NeTEx

To compare lines between the two formats, both datasets are reduced to a single
column of **public line labels**, the short codes that passengers see, such as
"1", "31", or "NW450".

Both label columns are kept as plain text and stripped of any leading or trailing
spaces. This is important because labels like "01" and "1" are technically different
and should not be silently merged.

The two label sets are then compared using set operations: labels that appear in
both datasets are counted as matches, while labels only in GTFS or only in NeTEx
are counted separately. This gives a match rate from both sides.

In [43]:
# Line-level GTFS–NeTEx comparison by public label
gtfs_line_compare  = gtfs_lines.copy()
netex_line_compare = netex_lines_dedup.copy()

# Keep labels as strings and strip spaces
# Important: do not convert to numeric (labels like "001" and "01" must keep leading zeros)
gtfs_line_compare["line_label_compare"]  = gtfs_line_compare["gtfs_line_label"].astype(str).str.strip()
netex_line_compare["line_label_compare"] = netex_line_compare["netex_line_label"].astype(str).str.strip()

gtfs_labels   = set(gtfs_line_compare["line_label_compare"])
netex_labels  = set(netex_line_compare["line_label_compare"])
matched_labels     = gtfs_labels & netex_labels
gtfs_only_labels   = gtfs_labels - netex_labels
netex_only_labels  = netex_labels - gtfs_labels

gtfs_match_rate  = round(len(matched_labels) / len(gtfs_labels)  * 100, 2) if gtfs_labels  else 0.0
netex_match_rate = round(len(matched_labels) / len(netex_labels) * 100, 2) if netex_labels else 0.0

line_match_summary = pd.DataFrame({
    "metric": [
        "GTFS unique line labels",
        "NeTEx unique public codes",
        "matched labels",
        "GTFS-only labels",
        "NeTEx-only labels",
        "GTFS-side match rate (%)",
        "NeTEx-side match rate (%)"
    ],
    "value": [
        len(gtfs_labels),
        len(netex_labels),
        len(matched_labels),
        len(gtfs_only_labels),
        len(netex_only_labels),
        gtfs_match_rate,
        netex_match_rate
    ]
})
display(line_match_summary)

,metric,value
0,GTFS unique line labels,2460.00
1,NeTEx unique public codes,2461.00
2,matched labels,2460.00
3,GTFS-only labels,0.00
4,NeTEx-only labels,1.00
5,GTFS-side match rate (%),100.00
6,NeTEx-side match rate (%),99.96


In [44]:
# Example matched and unmatched line labels

print("Example matched labels:")
display(
    gtfs_line_compare[
        gtfs_line_compare["line_label_compare"].isin(matched_labels)
    ][["gtfs_line_label", "number_of_route_rows", "route_types", "example_route_long_name"]]
    .head(20)
)

print("Example GTFS-only labels:")
display(
    gtfs_line_compare[
        gtfs_line_compare["line_label_compare"].isin(gtfs_only_labels)
    ][["gtfs_line_label", "number_of_route_rows", "route_types", "example_route_long_name"]]
    .head(20)
)

print("Example NeTEx-only labels:")
display(
    netex_line_compare[
        netex_line_compare["line_label_compare"].isin(netex_only_labels)
    ][["netex_line_label", "number_of_line_rows", "transport_modes", "example_line_name"]]
    .head(20)
)

Example matched labels:


,gtfs_line_label,number_of_route_rows,route_types,example_route_long_name
0,001,1,[1008],Hyke Arendalsuka
1,01,1,[704],Horten-Tønsberg-Sandefjord
2,011,1,[704],Larvik-Tønsberg
3,02,4,"[704, 712]",Horten vgs
4,021,1,[704],Tønsberg-Skoppum-Holmestrand
5,022,3,"[704, 712]",Greveskogen-Tønsberg-V.Ende
6,023,1,[704],Nykirke-Horten-Tønsberg
7,1,11,"[401, 701, 704, 901]",Ranheim - Strindheim - sentrum - Tiller - Heim...
8,10,9,"[701, 704]",Sykehuset - Kvadraturen
9,100,18,"[701, 702, 704, 712]",Arendal-Kristiansand


Example GTFS-only labels:


,gtfs_line_label,number_of_route_rows,route_types,example_route_long_name


Example NeTEx-only labels:


,netex_line_label,number_of_line_rows,transport_modes,example_line_name
1863,Voss Gondol,1,cableway,Voss Gondol


## Comment

The line-level comparison gives a near-perfect result.

Of 2,460 unique GTFS line labels, all 2,460 matched a NeTEx public code, giving a
**100% GTFS-side match rate**. NeTEx contains one additional label that GTFS does not,
bringing the NeTEx-side match rate to **99.96%**.

The single NeTEx-only label is **"Voss Gondol"**, a cableway service. This is a
non-standard transport mode that is likely not represented in GTFS, which explains
why it has no counterpart on the GTFS side.

The GTFS-only table is empty, confirming that every GTFS line has a matching NeTEx entry.

The sample of matched labels also shows that the label format is consistent across both
formats: leading zeros are preserved (e.g. "001", "01", "02"), and the same label can
cover multiple route rows or transport modes within the same dataset, which reflects
Norway's multimodal structure.


## Check transport modes for matched line labels


In [45]:
# Compare transport mode context for matched line labels
gtfs_line_modes = routes[routes["route_short_name"].notna()].copy()
gtfs_line_modes["route_short_name"] = gtfs_line_modes["route_short_name"].str.strip()
gtfs_line_modes["route_type"] = gtfs_line_modes["route_type"].astype(int)
gtfs_line_modes["gtfs_transport_mode"] = gtfs_line_modes["route_type"].apply(classify_route_type)
gtfs_line_modes = (
    gtfs_line_modes
    .groupby("route_short_name", as_index=False)
    .agg(
        gtfs_route_types=(
            "route_type",
            lambda x: ", ".join(map(str, sorted(x.dropna().unique())))
        ),
        gtfs_transport_modes=(
            "gtfs_transport_mode",
            lambda x: ", ".join(sorted(x.dropna().unique()))
        )
    )
    .rename(columns={"route_short_name": "line_label_compare"})
)

netex_line_modes = netex_line_compare[
    ["line_label_compare", "transport_modes", "example_line_name"]
].copy()

line_mode_comparison = gtfs_line_modes.merge(
    netex_line_modes,
    on="line_label_compare",
    how="inner"
)
display(line_mode_comparison.head(30))

,line_label_compare,gtfs_route_types,gtfs_transport_modes,transport_modes,example_line_name
0,001,1008,Ferry / water transport,water,Hyke Arendalsuka
1,01,704,Bus,bus,Horten-Tønsberg-Sandefjord
2,011,704,Bus,bus,Larvik-Tønsberg
3,02,"704, 712",Bus,bus,Tjøme-Tønsberg-Horten-Holmestr
4,021,704,Bus,bus,Tønsberg-Skoppum-Holmestrand
5,022,"704, 712",Bus,bus,Greveskogen-Tønsberg-V.Ende
6,023,704,Bus,bus,Nykirke-Horten-Tønsberg
7,1,"401, 701, 704, 901","Bus, Metro / urban rail, Tram","bus, metro, tram",Håkvik - Sentrum - Universitetssykehuset
8,10,"701, 704",Bus,bus,Vikerkilen-Fredrikstad
9,100,"701, 702, 704, 712",Bus,bus,Teie torv-Vestfjordveien


## Comment

The transport mode comparison shows strong alignment between GTFS and NeTEx across all
matched line labels.

Bus lines are the most common in both datasets and consistently labelled as such on both
sides. Ferry routes, identifiable by labels in the 1000–1031 range, are classified as
"Ferry / water transport" in GTFS and "water" in NeTEx, which refer to the same mode
despite the different terminology.

One interesting case is label **"1"**, which carries four different GTFS route type codes
(401, 701, 704, 901), mapping to bus, metro, and tram. NeTEx agrees on all three modes.
This reflects Norway's multimodal structure, where the same public line label can cover
services operated by different agencies or vehicle types across different regions.

Overall, the mode classification is consistent between the two formats for all 30 sampled
labels.

In [46]:
# Side-by-side table of matched GTFS and NeTEx line labels
line_label_table = gtfs_line_compare[
    gtfs_line_compare["line_label_compare"].isin(matched_labels)
][["gtfs_line_label", "line_label_compare"]].merge(
    netex_line_compare[
        netex_line_compare["line_label_compare"].isin(matched_labels)
    ][["netex_line_label", "line_label_compare"]],
    on="line_label_compare"
)[["gtfs_line_label", "netex_line_label"]].drop_duplicates().sort_values("gtfs_line_label").reset_index(drop=True)

display(line_label_table.head(30))

,gtfs_line_label,netex_line_label
0,001,001
1,01,01
2,011,011
3,02,02
4,021,021
5,022,022
6,023,023
7,1,1
8,10,10
9,100,100


## 3.3 Calendar comparison

After the station and line comparisons, I now start the calendar-level comparison.

GTFS uses `service_id` to describe service validity.  
NeTEx uses `daytype_ref` for the extracted active dates.

Before comparing the full active-day patterns, I first check whether the GTFS `service_id` values and NeTEx `daytype_ref` values refer to the same identifiers.

## Calendar-level comparison: GTFS vs NeTEx

Unlike stations and lines, the calendar comparison cannot be done by a simple label match.
GTFS uses `service_id` values and NeTEx uses `DayType` references and these are different
identifier systems, but in Norway both are based on the shared NSR identifier scheme,
so partial overlap is possible and meaningful to check.

This cell compares the two ID sets directly to measure how much overlap exists before
moving to the active dates comparison. A partial match is expected: NeTEx models calendar
at a finer granularity than GTFS, meaning one GTFS service may correspond to multiple
NeTEx DayTypes, and vice versa.

In [47]:
# Calendar-level ID coverage between GTFS service_id and NeTEx daytype_ref

gtfs_calendar_ids = set(gtfs_calendar_summary["service_id"].dropna())
netex_calendar_ids = set(netex_calendar_summary["daytype_ref"].dropna())

matched_calendar_ids = gtfs_calendar_ids & netex_calendar_ids
gtfs_only_calendar_ids = gtfs_calendar_ids - netex_calendar_ids
netex_only_calendar_ids = netex_calendar_ids - gtfs_calendar_ids

calendar_id_summary = pd.DataFrame({
    "metric": [
        "GTFS service_id values",
        "NeTEx daytype_ref values",
        "matched calendar IDs",
        "GTFS-only calendar IDs",
        "NeTEx-only calendar IDs",
        "GTFS-side ID match rate (%)",
        "NeTEx-side ID match rate (%)"
    ],
    "value": [
        len(gtfs_calendar_ids),
        len(netex_calendar_ids),
        len(matched_calendar_ids),
        len(gtfs_only_calendar_ids),
        len(netex_only_calendar_ids),
        round(len(matched_calendar_ids) / len(gtfs_calendar_ids) * 100, 2) if gtfs_calendar_ids else 0.0,
        round(len(matched_calendar_ids) / len(netex_calendar_ids) * 100, 2) if netex_calendar_ids else 0.0
    ]
})

display(calendar_id_summary)

print("Example matched calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(matched_calendar_ids))[:20]}))

print("Example GTFS-only calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(gtfs_only_calendar_ids))[:20]}))

print("Example NeTEx-only calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(netex_only_calendar_ids))[:20]}))

,metric,value
0,GTFS service_id values,9854.00
1,NeTEx daytype_ref values,15524.00
2,matched calendar IDs,6635.00
3,GTFS-only calendar IDs,3219.00
4,NeTEx-only calendar IDs,8889.00
5,GTFS-side ID match rate (%),67.33
6,NeTEx-side ID match rate (%),42.74


Example matched calendar IDs:


,calendar_id
0,AKT:DayType:100_1
1,AKT:DayType:101_1
2,AKT:DayType:102_1
3,AKT:DayType:103_1
4,AKT:DayType:104_1
5,AKT:DayType:105_1
6,AKT:DayType:106_1
7,AKT:DayType:107_1
8,AKT:DayType:108_1
9,AKT:DayType:109_1


Example GTFS-only calendar IDs:


,calendar_id
0,AKT:DayType:118_1-133_1
1,ASH:DayType:6bc213fc-d314-4917-a7d1-fe9744fca221
2,ASH:DayType:6bc213fc-d314-4917-a7d1-fe9744fca2...
3,ASH:DayType:8e7ea1d1-64e5-4fd3-9b84-c9f93a25563b
4,ASH:DayType:c0c23b27-b321-4834-bd1d-ca6d541c10e0
5,ATU:DayType:3ba7a127-0217-4f8e-822a-b6fd3fd38f...
6,ATU:DayType:3ba7a127-0217-4f8e-822a-b6fd3fd38f...
7,ATU:DayType:3ba7a127-0217-4f8e-822a-b6fd3fd38f...
8,ATU:DayType:3ba7a127-0217-4f8e-822a-b6fd3fd38f...
9,ATU:DayType:3ba7a127-0217-4f8e-822a-b6fd3fd38f...


Example NeTEx-only calendar IDs:


,calendar_id
0,ATB:DayType:19b85008-7c92-59c8-b965-dc9789704510
1,ATB:DayType:1b645389-2473-5467-9073-72d45eb05abc
2,ATB:DayType:1e9f43e7-e719-5ff0-a787-1e13e9ab1371
3,ATB:DayType:1f02f98c-f775-59e0-8e92-52ff5972dce7
4,ATB:DayType:2219fecb-861a-582b-a270-6e49a065c8d1
5,ATB:DayType:24a8d73e-5826-513a-aca9-54042e9d9f71
6,ATB:DayType:299fb752-7943-5d46-8472-ce01b4214088
7,ATB:DayType:2f85f7e8-d92e-5594-a8db-92ac11426e34
8,ATB:DayType:2fab7143-0bf4-5f76-8993-7ce09c599b1a
9,ATB:DayType:356a192b-7913-504c-9457-4d18c28d46e6


## Comment

The ID comparison shows a **67% GTFS-side match rate** and a **43% NeTEx-side match rate**,
meaning the two formats share a common ID space but do not fully overlap.

The calendar ID shows only a partial overlap between GTFS `service_id` values and NeTEx `daytype_ref` values.

GTFS has 9,854 calendar IDs, while NeTEx has 15,524 day type references. Out of these, 6,635 IDs appear on both sides. This gives a GTFS-side ID match rate of 67.33% and a NeTEx-side ID match rate of 42.74%.

Some IDs match directly, especially simple IDs such as `AKT:DayType:100_1`, but many other IDs exist only on one side. This is expected because GTFS and NeTEx can use different identifier structures for calendar validity.

The real calendar comparison should therefore continue with active-day patterns, not only identifier matching.

## Calendar-level comparison for matched IDs

The previous step showed that some GTFS `service_id` values also appear as NeTEx `daytype_ref` values.

I now compare the actual active dates for these matched IDs.  
This checks whether a calendar ID that exists on both sides also describes the same service days.

This is still only the comparison for directly matched IDs. If some IDs do not match by name, they can still represent similar patterns, but that would require a later pattern-level comparison.

In [48]:
# Compare active dates for matched GTFS service_id / NeTEx daytype_ref values

# Normalize GTFS active date sets
gtfs_dates_by_id = {
    service_id: set(pd.to_datetime(list(dates)).normalize())
    for service_id, dates in gtfs_active_dates.items()
}

# Normalize NeTEx active date sets
netex_dates_by_id = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .assign(active_date=lambda df: pd.to_datetime(df["active_date"]).dt.normalize())
    .groupby("daytype_ref")["active_date"]
    .apply(set)
    .to_dict()
)

calendar_pattern_rows = []
for calendar_id in matched_calendar_ids:
    gtfs_dates  = gtfs_dates_by_id.get(calendar_id, set())
    netex_dates = netex_dates_by_id.get(calendar_id, set())
    gtfs_only_dates  = gtfs_dates - netex_dates
    netex_only_dates = netex_dates - gtfs_dates
    calendar_pattern_rows.append({
        "calendar_id":         calendar_id,
        "gtfs_active_days":    len(gtfs_dates),
        "netex_active_days":   len(netex_dates),
        "shared_active_days":  len(gtfs_dates & netex_dates),
        "gtfs_only_days":      len(gtfs_only_dates),
        "netex_only_days":     len(netex_only_dates),
        # Guard against two empty sets falsely comparing as equal
        "exact_pattern_match": gtfs_dates == netex_dates and len(gtfs_dates) > 0 and len(netex_dates) > 0
    })

calendar_pattern_comparison = pd.DataFrame(calendar_pattern_rows)

calendar_pattern_summary = pd.DataFrame({
    "metric": [
        "matched calendar IDs checked",
        "exact active-date pattern matches",
        "active-date pattern mismatches",
        "exact match rate (%)"
    ],
    "value": [
        len(calendar_pattern_comparison),
        calendar_pattern_comparison["exact_pattern_match"].sum(),
        (~calendar_pattern_comparison["exact_pattern_match"]).sum(),
        round(calendar_pattern_comparison["exact_pattern_match"].mean() * 100, 2)
    ]
})
display(calendar_pattern_summary)

display(
    calendar_pattern_comparison
    .sort_values(
        ["exact_pattern_match", "gtfs_only_days", "netex_only_days"],
        ascending=[True, False, False]
    )
    .head(20)
)

,metric,value
0,matched calendar IDs checked,6635.00
1,exact active-date pattern matches,2802.00
2,active-date pattern mismatches,3833.00
3,exact match rate (%),42.23


,calendar_id,gtfs_active_days,netex_active_days,shared_active_days,gtfs_only_days,netex_only_days,exact_pattern_match
6337,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,66,549,66,0,483,False
6398,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,69,549,69,0,480,False
4599,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,71,549,71,0,478,False
4385,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,72,549,72,0,477,False
4277,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,73,549,73,0,476,False
742,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,77,549,77,0,472,False
5164,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,77,549,77,0,472,False
6388,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,77,549,77,0,472,False
421,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,78,549,78,0,471,False
2063,FIN:DayType:Cal2026050846067-Feb_Sat_15-Dec_Th...,78,549,78,0,471,False


## Comment

The direct calendar-ID comparison gives a mixed result.

Out of 6,635 calendar IDs that exist on both sides, 2,802 have exactly the same active-day pattern in GTFS and NeTEx. This gives an exact match rate of 42.23%.

Many mismatches are not caused by GTFS having extra days. In the examples, GTFS has 0 GTFS-only days, while NeTEx has many additional days. This means that for these IDs, the GTFS active dates are included in the NeTEx active dates, but NeTEx describes a broader validity period.

## Calendar-level pattern comparison within the shared date window

The direct ID comparison showed that some NeTEx calendar IDs cover a broader period than the matching GTFS service IDs.

To make the comparison fairer, I now restrict both sides to the same shared date window.  
Then I convert each GTFS `service_id` and each NeTEx `daytype_ref` into an active-day pattern over that window.

This comparison does not depend on matching IDs. It checks whether the same active-day patterns exist in both formats.

Steps:

- normalize GTFS and NeTEx active dates
- find the earliest and latest active dates on each side
- define the shared comparison window as the overlap between both date ranges

In [49]:
# Calendar-level pattern comparison within shared date window
# (gtfs_dates_by_id and netex_dates_by_id were already computed in the previous
# step and are reused here — nothing between the two cells changes their inputs)

# Define shared comparison window across all IDs in both feeds
all_gtfs_dates  = [d for dates in gtfs_dates_by_id.values()  for d in dates]
all_netex_dates = [d for dates in netex_dates_by_id.values() for d in dates]

gtfs_min_date  = min(all_gtfs_dates)
gtfs_max_date  = max(all_gtfs_dates)
netex_min_date = min(all_netex_dates)
netex_max_date = max(all_netex_dates)

shared_start_date = max(gtfs_min_date, netex_min_date)
shared_end_date   = min(gtfs_max_date, netex_max_date)
shared_dates      = pd.date_range(shared_start_date, shared_end_date, freq="D")

shared_window_summary = pd.DataFrame({
    "metric": [
        "GTFS earliest active date",
        "GTFS latest active date",
        "NeTEx earliest active date",
        "NeTEx latest active date",
        "shared start date",
        "shared end date",
        "number of days in shared window"
    ],
    "value": [
        gtfs_min_date,
        gtfs_max_date,
        netex_min_date,
        netex_max_date,
        shared_start_date,
        shared_end_date,
        len(shared_dates)
    ]
})
display(shared_window_summary)

,metric,value
0,GTFS earliest active date,2019-04-15 00:00:00
1,GTFS latest active date,2027-06-20 00:00:00
2,NeTEx earliest active date,2019-04-13 00:00:00
3,NeTEx latest active date,2027-12-31 00:00:00
4,shared start date,2019-04-15 00:00:00
5,shared end date,2027-06-20 00:00:00
6,number of days in shared window,2989


## Output

The shared calendar comparison window was defined successfully.

GTFS active dates range from 2019-04-15 to 2027-06-20, while NeTEx active dates range from 2019-04-13 to 2027-12-31.

The overlapping period between both feeds is therefore from 2019-04-15 to 2027-06-20. This gives a shared comparison window of 2,989 days.

This shared window will be used for the pattern-level comparison, so both GTFS and NeTEx are evaluated over the same date range.

## Build calendar activity patterns in the shared window

I now convert each GTFS `service_id` and each NeTEx `daytype_ref` into an activity pattern over the shared date window.

Each pattern is a string of 0s and 1s:
- `1` means the service is active on that date
- `0` means the service is not active on that date

Calendar IDs with no active dates inside the shared window are kept in the first summary, but they are excluded from the final pattern comparison because an all-zero pattern does not represent an actual service pattern.

In [50]:
# Build activity patterns over the shared comparison window

def build_activity_pattern(active_dates, comparison_dates):
    active_dates = set(pd.to_datetime(list(active_dates)).normalize())
    return "".join("1" if d in active_dates else "0" for d in comparison_dates)

# Build GTFS calendar patterns
gtfs_pattern_rows = []

for service_id, active_dates in gtfs_dates_by_id.items():
    window_dates = {
        d for d in active_dates
        if shared_start_date <= d <= shared_end_date
    }

    gtfs_pattern_rows.append({
        "gtfs_service_id": service_id,
        "n_active_days_window": len(window_dates),
        "first_active_date_window": min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window": max(window_dates) if window_dates else pd.NaT,
        "activity_pattern": build_activity_pattern(window_dates, shared_dates)
    })

gtfs_calendar_patterns = pd.DataFrame(gtfs_pattern_rows)

# Build NeTEx calendar patterns
netex_pattern_rows = []

for daytype_ref, active_dates in netex_dates_by_id.items():
    window_dates = {
        d for d in active_dates
        if shared_start_date <= d <= shared_end_date
    }

    netex_pattern_rows.append({
        "netex_daytype_ref": daytype_ref,
        "n_active_days_window": len(window_dates),
        "first_active_date_window": min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window": max(window_dates) if window_dates else pd.NaT,
        "activity_pattern": build_activity_pattern(window_dates, shared_dates)
    })

netex_calendar_patterns = pd.DataFrame(netex_pattern_rows)

# Keep only calendars with at least one active date in the shared window
gtfs_calendar_patterns_active = gtfs_calendar_patterns[
    gtfs_calendar_patterns["n_active_days_window"] > 0
].copy()

netex_calendar_patterns_active = netex_calendar_patterns[
    netex_calendar_patterns["n_active_days_window"] > 0
].copy()

pattern_preparation_summary = pd.DataFrame({
    "metric": [
        "GTFS calendar rows",
        "GTFS rows with active days in shared window",
        "GTFS rows with zero active days in shared window",
        "GTFS unique activity patterns",
        "NeTEx calendar rows",
        "NeTEx rows with active days in shared window",
        "NeTEx rows with zero active days in shared window",
        "NeTEx unique activity patterns"
    ],
    "value": [
        len(gtfs_calendar_patterns),
        len(gtfs_calendar_patterns_active),
        (gtfs_calendar_patterns["n_active_days_window"] == 0).sum(),
        gtfs_calendar_patterns_active["activity_pattern"].nunique(),
        len(netex_calendar_patterns),
        len(netex_calendar_patterns_active),
        (netex_calendar_patterns["n_active_days_window"] == 0).sum(),
        netex_calendar_patterns_active["activity_pattern"].nunique()
    ]
})

display(pattern_preparation_summary)

print("Example GTFS calendar patterns:")
display(gtfs_calendar_patterns_active.head(10))

print("Example NeTEx calendar patterns:")
display(netex_calendar_patterns_active.head(10))

,metric,value
0,GTFS calendar rows,9854
1,GTFS rows with active days in shared window,9854
2,GTFS rows with zero active days in shared window,0
3,GTFS unique activity patterns,4791
4,NeTEx calendar rows,15524
5,NeTEx rows with active days in shared window,15524
6,NeTEx rows with zero active days in shared window,0
7,NeTEx unique activity patterns,1708


Example GTFS calendar patterns:


,gtfs_service_id,n_active_days_window,first_active_date_window,last_active_date_window,activity_pattern
0,AKT:DayType:100_1,27,2026-05-07,2026-10-01,0000000000000000000000000000000000000000000000...
1,AKT:DayType:101_1,24,2026-05-07,2026-09-25,0000000000000000000000000000000000000000000000...
2,AKT:DayType:102_1,48,2026-05-06,2026-09-25,0000000000000000000000000000000000000000000000...
3,AKT:DayType:105_1,48,2026-05-06,2026-09-25,0000000000000000000000000000000000000000000000...
4,AKT:DayType:106_1,83,2026-05-06,2026-10-03,0000000000000000000000000000000000000000000000...
5,AKT:DayType:107_1,9,2026-05-15,2026-10-02,0000000000000000000000000000000000000000000000...
6,AKT:DayType:108_1,37,2026-08-13,2026-10-02,0000000000000000000000000000000000000000000000...
7,AKT:DayType:109_1,32,2026-08-13,2026-09-25,0000000000000000000000000000000000000000000000...
8,AKT:DayType:10_1,51,2026-05-06,2026-09-25,0000000000000000000000000000000000000000000000...
9,AKT:DayType:110_1,6,2026-08-19,2026-09-23,0000000000000000000000000000000000000000000000...


Example NeTEx calendar patterns:


,netex_daytype_ref,n_active_days_window,first_active_date_window,last_active_date_window,activity_pattern
0,AKT:DayType:100_1,153,2026-05-06,2026-10-05,0000000000000000000000000000000000000000000000...
1,AKT:DayType:101_1,152,2026-05-02,2026-09-30,0000000000000000000000000000000000000000000000...
2,AKT:DayType:102_1,145,2026-05-06,2026-09-27,0000000000000000000000000000000000000000000000...
3,AKT:DayType:103_1,7,2026-05-15,2026-10-02,0000000000000000000000000000000000000000000000...
4,AKT:DayType:104_1,8,2026-05-15,2026-10-02,0000000000000000000000000000000000000000000000...
5,AKT:DayType:105_1,146,2026-05-05,2026-09-27,0000000000000000000000000000000000000000000000...
6,AKT:DayType:106_1,152,2026-05-06,2026-10-04,0000000000000000000000000000000000000000000000...
7,AKT:DayType:107_1,153,2026-05-09,2026-10-08,0000000000000000000000000000000000000000000000...
8,AKT:DayType:108_1,53,2026-08-13,2026-10-04,0000000000000000000000000000000000000000000000...
9,AKT:DayType:109_1,46,2026-08-13,2026-09-27,0000000000000000000000000000000000000000000000...


## Comment

The calendar activity patterns were created successfully for both formats.

All GTFS calendar rows and all NeTEx calendar rows have at least one active day inside the shared comparison window, so no rows had to be excluded.

GTFS has 9,854 calendar rows, which reduce to 4,791 unique activity patterns. NeTEx has 15,524 calendar rows, which reduce to 1,708 unique activity patterns.

This shows that both formats contain many repeated calendar structures. 

## Compare unique calendar activity patterns

The previous step converted each GTFS `service_id` and each NeTEx `daytype_ref` into an activity pattern over the same shared date window.

I now compare the unique activity patterns directly.  
This does not require GTFS and NeTEx calendar IDs to have the same name. It only checks whether the same active-day pattern exists in both formats.

In [51]:
# Compare unique GTFS and NeTEx activity patterns

# Prepare unique GTFS patterns
gtfs_unique_patterns = (
    gtfs_calendar_patterns_active
    .assign(pattern_active_days=lambda df: df["activity_pattern"].str.count("1"))
    .groupby("activity_pattern", as_index=False)
    .agg(
        number_of_gtfs_ids=("gtfs_service_id", "count"),
        example_gtfs_service_id=("gtfs_service_id", "first"),
        n_active_days_window=("pattern_active_days", "first"),
        first_active_date_window=("first_active_date_window", "first"),
        last_active_date_window=("last_active_date_window", "first")
    )
)

# Prepare unique NeTEx patterns
netex_unique_patterns = (
    netex_calendar_patterns_active
    .assign(pattern_active_days=lambda df: df["activity_pattern"].str.count("1"))
    .groupby("activity_pattern", as_index=False)
    .agg(
        number_of_netex_ids=("netex_daytype_ref", "count"),
        example_netex_daytype_ref=("netex_daytype_ref", "first"),
        n_active_days_window=("pattern_active_days", "first"),
        first_active_date_window=("first_active_date_window", "first"),
        last_active_date_window=("last_active_date_window", "first")
    )
)

# Compare pattern sets
gtfs_pattern_set = set(gtfs_unique_patterns["activity_pattern"])
netex_pattern_set = set(netex_unique_patterns["activity_pattern"])

matched_patterns = gtfs_pattern_set & netex_pattern_set
gtfs_only_patterns = gtfs_pattern_set - netex_pattern_set
netex_only_patterns = netex_pattern_set - gtfs_pattern_set

gtfs_pattern_match_rate = (
    round(len(matched_patterns) / len(gtfs_pattern_set) * 100, 2)
    if gtfs_pattern_set else 0.0
)

netex_pattern_match_rate = (
    round(len(matched_patterns) / len(netex_pattern_set) * 100, 2)
    if netex_pattern_set else 0.0
)

pattern_match_summary = pd.DataFrame({
    "metric": [
        "GTFS unique activity patterns",
        "NeTEx unique activity patterns",
        "matched activity patterns",
        "GTFS-only activity patterns",
        "NeTEx-only activity patterns",
        "GTFS-side pattern match rate (%)",
        "NeTEx-side pattern match rate (%)"
    ],
    "value": [
        len(gtfs_pattern_set),
        len(netex_pattern_set),
        len(matched_patterns),
        len(gtfs_only_patterns),
        len(netex_only_patterns),
        gtfs_pattern_match_rate,
        netex_pattern_match_rate
    ]
})

display(pattern_match_summary)

,metric,value
0,GTFS unique activity patterns,4791.00
1,NeTEx unique activity patterns,1708.00
2,matched activity patterns,840.00
3,GTFS-only activity patterns,3951.00
4,NeTEx-only activity patterns,868.00
5,GTFS-side pattern match rate (%),17.53
6,NeTEx-side pattern match rate (%),49.18


In [52]:
# Example matched and unmatched calendar patterns

matched_pattern_examples = (
    gtfs_unique_patterns[gtfs_unique_patterns["activity_pattern"].isin(matched_patterns)]
    .merge(
        netex_unique_patterns[netex_unique_patterns["activity_pattern"].isin(matched_patterns)],
        on="activity_pattern",
        suffixes=("_gtfs", "_netex")
    )
)

print("Example matched activity patterns:")
display(
    matched_pattern_examples[
        [
            "example_gtfs_service_id",
            "example_netex_daytype_ref",
            "number_of_gtfs_ids",
            "number_of_netex_ids",
            "n_active_days_window_gtfs",
            "first_active_date_window_gtfs",
            "last_active_date_window_gtfs"
        ]
    ].head(20)
)

print("Example GTFS-only activity patterns:")
display(
    gtfs_unique_patterns[
        gtfs_unique_patterns["activity_pattern"].isin(gtfs_only_patterns)
    ][
        [
            "example_gtfs_service_id",
            "number_of_gtfs_ids",
            "n_active_days_window",
            "first_active_date_window",
            "last_active_date_window"
        ]
    ].head(20)
)

print("Example NeTEx-only activity patterns:")
display(
    netex_unique_patterns[
        netex_unique_patterns["activity_pattern"].isin(netex_only_patterns)
    ][
        [
            "example_netex_daytype_ref",
            "number_of_netex_ids",
            "n_active_days_window",
            "first_active_date_window",
            "last_active_date_window"
        ]
    ].head(20)
)

Example matched activity patterns:


,example_gtfs_service_id,example_netex_daytype_ref,number_of_gtfs_ids,number_of_netex_ids,n_active_days_window_gtfs,first_active_date_window_gtfs,last_active_date_window_gtfs
0,NOR:DayType:618,NOR:DayType:618,1,1,1,2027-01-01,2027-01-01
1,NOR:DayType:619,KOL:DayType:1185,2,2,1,2026-12-26,2026-12-26
2,KOL:DayType:1276,KOL:DayType:1276,2,2,1,2026-12-25,2026-12-25
3,NOR:OperatingDay:2026-12-24,KOL:DayType:1277,2,2,1,2026-12-24,2026-12-24
4,KOL:DayType:1289,KOL:DayType:1289,2,2,2,2026-12-24,2026-12-31
5,NOR:DayType:623,NOR:DayType:623,1,1,9,2026-12-24,2027-01-01
6,SOF:DayType:300,SOF:DayType:300,1,1,5,2026-11-01,2026-11-29
7,HUR:DayType:Autumn2026-10-30,HUR:DayType:Autumn2026-10-30,1,1,1,2026-10-30,2026-10-30
8,HUR:DayType:Autumn2026-10-28,HUR:DayType:Autumn2026-10-28,1,1,1,2026-10-28,2026-10-28
9,HUR:DayType:Autumn2026-10-27,HUR:DayType:Autumn2026-10-27,1,1,1,2026-10-27,2026-10-27


Example GTFS-only activity patterns:


,example_gtfs_service_id,number_of_gtfs_ids,n_active_days_window,first_active_date_window,last_active_date_window
5,BFO:DayType:3b67df67-ecde-4b8f-80a5-4a19e34fec3b,1,3,2026-12-24,2026-12-31
7,KOL:DayType:1278,1,6,2026-11-17,2026-12-22
8,KOL:DayType:1198,1,21,2026-11-16,2026-12-21
9,KOL:DayType:1199,1,27,2026-11-16,2026-12-22
10,TRO:DayType:260504112607711_990,1,7,2026-11-07,2026-12-19
11,NOR:OperatingDay:2026-11-05-2026-11-12-2026-11...,1,11,2026-11-05,2027-01-28
12,NOR:OperatingDay:2026-11-02-2026-11-04-2026-11...,1,26,2026-11-02,2027-01-27
13,NOR:OperatingDay:2026-11-02-2026-11-04-2026-11...,1,37,2026-11-02,2027-01-29
14,HUR:DayType:Winter,1,6,2026-11-02,2027-03-31
15,TRO:DayType:260504112607566_845,1,41,2026-11-02,2026-12-30


Example NeTEx-only activity patterns:


,example_netex_daytype_ref,number_of_netex_ids,n_active_days_window,first_active_date_window,last_active_date_window
3,VKT:DayType:Cal2026043049996-Mai_Mon_04-Dec_Th...,1,2,2026-12-25,2026-12-26
7,FIN:DayType:Cal2026050846067-Apr_Sun_19-Dec_Th...,19,1,2026-12-23,2026-12-23
8,KOL:DayType:1198,2,39,2026-11-14,2026-12-22
9,KOL:DayType:1278,1,48,2026-11-11,2026-12-28
11,TRO:DayType:260504112607711_990,1,55,2026-11-01,2026-12-25
12,HUR:DayType:Winter,1,151,2026-11-01,2027-03-31
13,TRO:DayType:260504112607566_845,1,61,2026-10-31,2026-12-30
23,VYX:DayType:793c5131a963101b49176079c086afc637...,1,133,2026-10-19,2027-02-28
26,SOF:DayType:113,1,48,2026-10-15,2026-12-01
28,SOF:DayType:123,1,48,2026-10-14,2026-11-30


## Comment

The pattern-level calendar comparison gives a weaker result than the station and line comparisons.

GTFS has 4,791 unique activity patterns, while NeTEx has 1,708 unique activity patterns. Out of these, 840 patterns are found in both formats.

This means that 17.53% of GTFS patterns also appear in NeTEx, and 49.18% of NeTEx patterns also appear in GTFS.

So the calendar level is only partly aligned. Some patterns match exactly, but many patterns exist only on one side. This is likely because GTFS and NeTEx group service days differently. NeTEx often uses broader operating periods, while GTFS can have more detailed or more fragmented service patterns.


In [53]:
# Inspect active-date distribution by year

gtfs_active_date_rows = []

for service_id, dates in gtfs_dates_by_id.items():
    for d in dates:
        gtfs_active_date_rows.append({
            "source": "GTFS",
            "calendar_id": service_id,
            "active_date": d
        })

gtfs_active_date_df = pd.DataFrame(gtfs_active_date_rows)

netex_active_date_df = (
    netex_active_dates[["daytype_ref", "active_date"]]
    .rename(columns={"daytype_ref": "calendar_id"})
    .copy()
)
netex_active_date_df["source"] = "NeTEx"

combined_active_dates = pd.concat(
    [gtfs_active_date_df, netex_active_date_df],
    ignore_index=True
)

combined_active_dates["active_date"] = pd.to_datetime(
    combined_active_dates["active_date"],
    errors="coerce"
).dt.normalize()

combined_active_dates["year"] = combined_active_dates["active_date"].dt.year

active_dates_by_year = (
    combined_active_dates
    .groupby(["source", "year"])
    .size()
    .reset_index(name="active_date_rows")
    .sort_values(["source", "year"])
)

display(active_dates_by_year)

,source,year,active_date_rows
0,GTFS,2019,118
1,GTFS,2020,191
2,GTFS,2021,11321
3,GTFS,2022,51722
4,GTFS,2023,40889
5,GTFS,2024,856
6,GTFS,2025,13010
7,GTFS,2026,350963
8,GTFS,2027,19312
9,NeTEx,2019,171


### Note on comparison method

The approach used for Norway differs from the other countries analysed so far.

In **Luxembourg** and **France**, NeTEx provides `ValidDayBits` which is a binary string
that directly encodes which days a service runs. The calendar comparison there was
straightforward: realign the bit strings to the same date window and compare them
character by character.

In **Norway**, `ValidDayBits` is not used at all. Instead, active dates have to be
reconstructed from scratch on both sides before any comparison is possible, from
`DayTypeAssignment` elements and `OperatingPeriod` date ranges on the NeTEx side,
and from weekly calendar rules plus exception dates on the GTFS side. This makes
the Norwegian calendar comparison significantly more complex than in the other countries.

## Calendar-level comparison: approach and results

Comparing calendars between GTFS and NeTEx is harder than comparing stations or lines,
because there is no shared label like a stop ID or a line number to match on directly.
Instead, both formats describe *when* a service runs using different internal structures,
and the comparison has to be done in several steps.

### Step 1 — ID overlap check
The first step checked whether any GTFS `service_id` values also appear as NeTEx
`DayType` references. Out of 9,854 GTFS service IDs and 15,524 NeTEx DayType references,
**6,635 IDs appeared on both sides** (67% of GTFS, 43% of NeTEx). The remaining IDs on each side use different naming
conventions and cannot be matched by name alone.

### Step 2 — Active date comparison for matched IDs
For the 6,635 shared IDs, the actual active dates were derived from both formats
by expanding calendar rules into individual day sets.

On the GTFS side, active dates were reconstructed from `calendar.txt` (weekly
rules applied across the validity period) and `calendar_dates.txt` (added and
removed exception dates). Services that only appear in `calendar_dates.txt` got
their active dates purely from the added exception dates.

On the NeTEx side, active dates were derived from `DayTypeAssignment` elements,
which either point to a specific date directly or to an `OperatingPeriod` with a
start and end date that was expanded into every individual day in that range.
Assignments marked as `IsAvailable = false` were excluded as explicitly cancelled days.

Both sides were then compared date by date for each shared ID. Out of the 6,635
matched IDs, **2,802 had exactly the same active dates on both sides**, giving an
exact match rate of **42.23%**.

### Step 3 — Pattern-level comparison over a shared window
To go beyond the matched IDs, all active date sets were converted into binary activity
patterns where each position represents one day in the shared
date window. This allows comparing the *shape* of a calendar, regardless of which ID it
carries.

GTFS produced **4,791 unique patterns** and NeTEx produced **1,708 unique patterns**.
Of these, **840 patterns appeared in both**, giving a GTFS-side match rate of **17.53%**
and a NeTEx-side match rate of **49.18%**.

## Conclusion: calendar-level MVD for Norway

The calendar comparison for Norway gives a more mixed result than stations and lines.

GTFS groups service days into service IDs,
while NeTEx defines a larger number of more granular DayTypes. This means NeTEx covers
more calendar entries than GTFS exposes, and many IDs and patterns exist only on one side.

The exact match rate of 42% for directly matched IDs, and the low pattern overlap of
17.5% on the GTFS side, show that the two formats do not describe calendar validity in
a fully consistent way for Norway. However, this is largely a structural difference
rather than missing data. Both formats contain the information needed to reconstruct
active service days, but they organise it at different levels of granularity.

The calendar MVD is therefore **partially confirmed** for Norway: the data needed to
determine when services run is present in both formats, but the alignment between them
is weaker than for stations and lines, and a direct comparison requires significant
reconstruction work on both sides.